# Подготовка данных и отбор признаков: полный индустриальный пайплайн

Цель ноутбука — не просто «почистить табличку», а сделать воспроизводимый feature engineering / feature selection pipeline для хакатонной задачи бинарной классификации с очень сильным дисбалансом.

Что делаем по этапам:

1. Загружаем `train.csv`, `test.csv`, `sample_submission.csv`.
2. Проверяем базовые свойства данных: размер, target rate, пропуски, нули, константные признаки, дубликаты.
3. Строим честную схему валидации с учётом дублей строк.
4. Удаляем явно бесполезные признаки: константные и почти константные.
5. Считаем несколько независимых оценок важности признаков:
   - корреляция с target;
   - mutual information;
   - LightGBM gain / split importance;
   - устойчивость важности по фолдам;
   - train/test drift.
6. Генерируем новые признаки:
   - sparse row statistics: сколько ненулевых, доля нулей, суммы, максимумы, нормы;
   - block statistics по группам исходных колонок;
   - top-k statistics по наиболее важным признакам;
   - peak flags: индикаторы попадания в «магические» значения;
   - frequency encoding для дискретных/повторяющихся значений;
   - OOF target encoding для низко- и среднекардинальных признаков;
   - pairwise interactions для топовых признаков;
   - PCA / SVD компоненты;
   - кластеры и расстояния до кластеров в PCA-пространстве.
7. Сохраняем несколько feature sets в `processed_data/`.
8. Быстро валидируем, какие наборы признаков дают прирост.

Главный принцип: **каждый рискованный шаг делается как эксперимент**, а не как догма. Особенно это касается удаления строк, target encoding и PCA.

## 0. Что важно понимать перед стартом

В этой задаче данные выглядят как анонимные числовые поведенческие признаки: много колонок, много нулей, мало положительного класса. Поэтому классические шаги вроде `StandardScaler` и «удалить выбросы по z-score» не являются главным источником качества.

Для бустингов важнее:

- убрать совсем мёртвые признаки;
- не сломать валидацию дублями;
- добавить row-level признаки: «насколько активна строка в целом»;
- найти повторяющиеся значения с сильным lift по target;
- собрать несколько разных feature set'ов для ансамбля.

`PCA` тоже попробуем, но не как замену исходных признаков. Для деревьев PCA часто не даёт чуда, потому что деревья сами хорошо режут исходные признаки. Но PCA/SVD может помочь как дополнительный взгляд на структуру строки, особенно для линейных моделей, нейросетей и stacking/blending.

In [1]:
# Базовые настройки ноутбука

from __future__ import annotations

import gc
import json
import math
import os
import sys
import time
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import StratifiedKFold, train_test_split
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.impute import SimpleImputer

try:
    import scipy.sparse as sp
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Основные переключатели. Если нужно ускорить запуск — выключай тяжёлые блоки.
RUN_MUTUAL_INFO = True
RUN_LGB_IMPORTANCE = True
RUN_PEAK_FLAGS = True
RUN_FREQUENCY_ENCODING = True
RUN_TARGET_ENCODING = True
RUN_INTERACTIONS = True
RUN_PCA_SVD = True
RUN_KMEANS_ON_PCA = True
RUN_BENCHMARK = True

# Ограничения по вычислениям.
N_SPLITS = 5
MI_SAMPLE_SIZE = 80_000
SPEARMAN_SAMPLE_SIZE = 80_000
LGB_IMPORTANCE_TIME_LIMIT_MINUTES = 60  # ориентир; LightGBM сам не ограничивается по времени, но модели тут небольшие
PCA_N_COMPONENTS = 32
SVD_N_COMPONENTS = 32
KMEANS_CLUSTERS = 16

# Тонкая настройка feature engineering.
LOW_VARIANCE_TOP_SHARE_THRESHOLD = 0.999
DRIFT_Z_THRESHOLD_TO_FLAG = 5.0
FREQ_MAX_NUNIQUE = 2_000
FREQ_TOP_N = 150
TARGET_ENCODING_TOP_N = 80
INTERACTION_TOP_N = 14
PEAK_TOP_FEATURES = 250
PEAK_MIN_POS_COUNT = 6
PEAK_MIN_TOTAL_COUNT = 10
PEAK_MIN_LIFT = 5.0

# Для frequency encoding: True = считаем частоты на train+test, потому что это unsupervised и часто помогает на соревнованиях.
# В чистом production-пайплайне обычно ставят False и считают только на train.
TRANSDUCTIVE_FREQ_ENCODING = True

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

## 1. Поиск файлов и загрузка данных

Ноутбук ищет данные в стандартных местах: `data/`, `raw_data/`, `ml_csvs/`, корень проекта и родительская папка. Это сделано, чтобы его можно было положить в `modeling/` и запустить из корня проекта без ручной переписки путей.

In [2]:
def find_first_existing_file(names: Sequence[str], search_dirs: Sequence[Path]) -> Optional[Path]:
    for d in search_dirs:
        for name in names:
            p = d / name
            if p.exists():
                return p.resolve()
    return None

PROJECT_ROOT = Path.cwd().resolve()
SEARCH_DIRS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "raw_data",
    PROJECT_ROOT / "processed_data",
    PROJECT_ROOT / "ml_csvs",
    PROJECT_ROOT.parent,
    PROJECT_ROOT.parent / "data",
    PROJECT_ROOT.parent / "raw_data",
    PROJECT_ROOT.parent / "ml_csvs",
    Path("/mnt/data/ml_csvs"),  # для среды ChatGPT; на твоём маке просто проигнорируется
    Path("/mnt/data"),
]

TRAIN_PATH = find_first_existing_file(["train.csv"], SEARCH_DIRS)
TEST_PATH = find_first_existing_file(["test.csv"], SEARCH_DIRS)
SAMPLE_SUB_PATH = find_first_existing_file(["sample_submission.csv", "submission.csv"], SEARCH_DIRS)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)

assert TRAIN_PATH is not None, "Не нашёл train.csv. Положи файл в data/train.csv или запусти ноутбук из корня проекта."
assert TEST_PATH is not None, "Не нашёл test.csv. Положи файл в data/test.csv или запусти ноутбук из корня проекта."

PROJECT_ROOT: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling
TRAIN_PATH: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data/train.csv
TEST_PATH: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/data/test.csv
SAMPLE_SUB_PATH: None


In [3]:
def build_dtype_map(csv_path: Path) -> Dict[str, str]:
    # Читаем только header и задаём экономные dtype для загрузки CSV.
    cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    dtype_map = {}
    for c in cols:
        if c == "index":
            dtype_map[c] = "int64"
        elif c == "target":
            dtype_map[c] = "int8"
        elif c.startswith("feature_"):
            dtype_map[c] = "float32"
    return dtype_map


def read_main_csv(path: Path) -> pd.DataFrame:
    t0 = time.time()
    df = pd.read_csv(path, dtype=build_dtype_map(path))
    mem_gb = df.memory_usage(deep=True).sum() / 1024**3
    print(f"Loaded {path.name}: shape={df.shape}, memory={mem_gb:.2f} GB, time={time.time() - t0:.1f}s")
    return df

train = read_main_csv(TRAIN_PATH)
test = read_main_csv(TEST_PATH)

sample_submission = None
if SAMPLE_SUB_PATH is not None:
    sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
    print("sample_submission:", sample_submission.shape)

ID_COL = "index"
TARGET_COL = "target"

feature_cols = [c for c in train.columns if c.startswith("feature_")]
assert TARGET_COL in train.columns
assert ID_COL in train.columns
assert ID_COL in test.columns

print("n_features:", len(feature_cols))
print("train rows:", len(train), "test rows:", len(test))

Loaded train.csv: shape=(247972, 1369), memory=1.26 GB, time=9.6s
Loaded test.csv: shape=(106274, 1368), memory=0.54 GB, time=3.8s
n_features: 1367
train rows: 247972 test rows: 106274


## 2. Базовая диагностика

Смотрим, что за задача: размер, дисбаланс target, пропуски, нули, типы признаков. Это не «просто формальность»: от этого зависит, нужны ли нам импьютеры, скейлеры, undersampling, sparse-признаки и т.д.

In [4]:
display(train.head())
display(test.head())

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Feature count:", len(feature_cols))

vc = train[TARGET_COL].value_counts().sort_index()
print("\nTarget counts:")
print(vc)
print("\nTarget rate:", train[TARGET_COL].mean())
print("Class imbalance neg/pos:", (vc.get(0, 0) / max(vc.get(1, 1), 1)))

missing_train = train[feature_cols].isna().sum().sum()
missing_test = test[feature_cols].isna().sum().sum()
print("\nTotal missing values in train features:", missing_train)
print("Total missing values in test features:", missing_test)

,index,target,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,...,feature_1307,feature_1308,feature_1309,feature_1310,feature_1311,feature_1312,feature_1313,feature_1314,feature_1315,feature_1316,feature_1317,feature_1318,feature_1319,feature_1320,feature_1321,feature_1322,feature_1323,feature_1324,feature_1325,feature_1326,feature_1327,feature_1328,feature_1329,feature_1330,feature_1331,feature_1332,feature_1333,feature_1334,feature_1335,feature_1336,feature_1337,feature_1338,feature_1339,feature_1340,feature_1341,feature_1342,feature_1343,feature_1344,feature_1345,feature_1346,feature_1347,feature_1348,feature_1349,feature_1350,feature_1351,feature_1352,feature_1353,feature_1354,feature_1355,feature_1356,feature_1357,feature_1358,feature_1359,feature_1360,feature_1361,feature_1362,feature_1363,feature_1364,feature_1365,feature_1366
0,239134,0,0.5,0.5,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.680302,0.767927,0.917902,0.527302,0.428348,0.466657,0.548376,0.815871,0.806104,0.729981,0.247545,0.604350,0.319196,0.717840,0.727683,0.769678,0.685146,0.582802,0.680368,0.890976,0.719449,0.635555,0.622619,0.489604,0.520638,0.460173,0.440582,0.524772,0.447167,0.480903,0.444122,0.735702,0.902803,0.395097,0.736835,0.490412,0.596536,0.904454,0.732110,0.0,0.011896,5.437209,39.134056,0.673629,1.419289,5.696633,15.822751,0.000508,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
1,234708,0,0.5,0.5,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.680302,0.767927,0.917902,0.527302,0.428348,0.466657,0.548376,0.815871,0.806104,0.729981,0.247545,0.604350,0.319196,0.717840,0.727683,0.769678,0.685146,0.582802,0.680368,0.890976,0.719449,0.635555,0.622619,0.489604,0.520638,0.460173,0.440582,0.524772,0.447167,0.480903,0.444122,0.735702,0.902803,0.395097,0.736835,0.490412,0.596536,0.904454,0.732110,0.0,0.011896,5.437209,39.134056,0.673629,1.419289,5.696633,15.822751,0.000508,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
2,268300,0,0.5,0.5,1.0,1.0,1.0,3.0,1.0,4.0,0.0,9.0,0.794758,2.471037,1.010160,0.288224,0.051471,0.154412,0.930185,0.970914,1.047707,1.046696,0.302190,0.970542,0.429787,0.670482,0.659651,0.726351,0.667095,0.609957,0.654339,0.790576,0.674854,0.593292,0.494033,0.494028,0.336947,0.071942,0.071942,0.812248,0.071942,0.140814,0.075436,0.596417,0.845669,0.103440,0.598182,0.475128,0.501427,0.995018,0.709474,0.0,0.000000,522.000000,28.000000,0.000000,12.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,283077,0,0.5,0.5,1.0,1.0,1.0,3.0,1.0,0.0,3.0,8.0,0.917668,0.558128,0.972966,0.213058,0.098592,0.072165,1.113402,0.987482,0.986724,0.986697,0.291172,1.117952,0.455936,0.638321,0.609359,0.696601,0.616842,0.494184,0.629273,0.760018,0.614402,0.492812,0.405435,0.379545,0.270000,0.200000,0.100000,1.207500,0.100000,0.180000,0.090000,0.669705,1.043027

,index,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,...,feature_1307,feature_1308,feature_1309,feature_1310,feature_1311,feature_1312,feature_1313,feature_1314,feature_1315,feature_1316,feature_1317,feature_1318,feature_1319,feature_1320,feature_1321,feature_1322,feature_1323,feature_1324,feature_1325,feature_1326,feature_1327,feature_1328,feature_1329,feature_1330,feature_1331,feature_1332,feature_1333,feature_1334,feature_1335,feature_1336,feature_1337,feature_1338,feature_1339,feature_1340,feature_1341,feature_1342,feature_1343,feature_1344,feature_1345,feature_1346,feature_1347,feature_1348,feature_1349,feature_1350,feature_1351,feature_1352,feature_1353,feature_1354,feature_1355,feature_1356,feature_1357,feature_1358,feature_1359,feature_1360,feature_1361,feature_1362,feature_1363,feature_1364,feature_1365,feature_1366
0,194357,0.5,0.5,1.0,1.0,1.0,1.0,1.0,3.0,0.0,6.0,0.968182,0.714778,1.265306,0.393116,0.152174,0.152174,0.880435,0.999675,0.956950,0.898636,0.215609,0.646174,0.354864,0.864612,0.900893,0.922357,0.887152,0.835618,0.899584,0.908295,0.900525,0.877328,0.878003,0.851101,0.189474,0.105263,0.105263,0.847368,0.105263,0.189474,0.284211,0.729507,0.926831,0.234820,0.965565,0.848896,0.385638,1.034000,0.934553,0.0,0.0,0.0,341.0,0.0,14.0,0.0,10.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,313222,0.5,0.5,1.0,1.0,0.0,2.0,2.0,2.0,0.0,6.0,0.638919,2.277712,0.972322,0.645833,0.875000,0.875000,0.562500,1.028136,0.861230,0.592205,0.468825,0.561151,0.592105,0.801061,0.749033,0.809998,0.751962,0.626032,0.722613,0.906852,0.718453,0.759986,0.750598,0.542589,0.818182,0.909091,0.909091,0.522727,0.909091,0.818182,0.818182,0.610724,0.910857,0.144266,0.950112,0.573724,0.312500,0.966167,0.750587,0.0,0.0,1.0,9.0,0.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,321873,0.5,0.5,1.0,1.0,0.0,3.0,1.0,1.0,0.0,5.0,0.680302,0.767927,0.917902,0.527302,0.428348,0.466657,0.548376,0.815871,0.806104,0.729981,0.247545,0.604350,0.319196,0.717840,0.727683,0.769678,0.685146,0.582802,0.680368,0.890976,0.719449,0.635555,0.622619,0.489604,0.520638,0.460173,0.440582,0.524772,0.447167,0.480903,0.444122,0.735702,0.902803,0.395097,0.736835,0.490412,0.596536,0.904454,0.732110,0.0,0.0,0.0,229.0,0.0,0.0,0.0,6.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,118689,0.5,0.5,1.0,1.0,1.0,3.0,2.0,3.0,0.0,9.0,0.684927,0.852196,0.959507,0.218927,0.118644,0.118644,0.610170,1.012567,1.025495,0.930575,0.387470,0.982014,0.470395,0.479265,0.554413,0.628606,0.482222,0.372899,0.472783,0.889012,0.570063,0.497831,0.452277,0.293005,0.446281,0.082645,0.082645,0.712810,0.082645,0.074380,0.074380,0.863935,0.796311,0.217637,0.245532,0.299714,0.241228,0.975725,0.587085,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

Train shape: (247972, 1369)
Test shape: (106274, 1368)
Feature count: 1367

Target counts:
target
0    244626
1      3346
Name: count, dtype: int64

Target rate: 0.013493458938912458
Class imbalance neg/pos: 73.10998206814106

Total missing values in train features: 0
Total missing values in test features: 0


In [5]:
# Быстрый профиль нулей и уникальных значений.
# Для high-dimensional sparse-like данных это намного информативнее, чем обычный describe().

zero_frac_train = train[feature_cols].eq(0).mean()
zero_frac_test = test[feature_cols].eq(0).mean()
nunique_train = train[feature_cols].nunique(dropna=False)

basic_profile = pd.DataFrame({
    "zero_frac_train": zero_frac_train,
    "zero_frac_test": zero_frac_test,
    "zero_frac_delta": (zero_frac_train - zero_frac_test).abs(),
    "nunique_train": nunique_train,
    "mean_train": train[feature_cols].mean(),
    "std_train": train[feature_cols].std(),
    "min_train": train[feature_cols].min(),
    "max_train": train[feature_cols].max(),
})

print("Zero fraction summary:")
display(basic_profile["zero_frac_train"].describe())

print("Most sparse features:")
display(basic_profile.sort_values("zero_frac_train", ascending=False).head(20))

print("Lowest cardinality features:")
display(basic_profile.sort_values("nunique_train").head(20))

Zero fraction summary:


count    1367.000000
mean        0.733958
std         0.259026
min         0.000000
25%         0.704959
50%         0.844144
75%         0.903239
max         1.000000
Name: zero_frac_train, dtype: float64

Most sparse features:


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,std_train,min_train,max_train
feature_49,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.0,0.000000
feature_1064,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.0,0.000000
feature_1057,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.0,0.000000
feature_1363,0.965258,0.965344,0.000086,2,0.034742,0.183126,0.0,1.000000
feature_697,0.903796,0.904774,0.000979,2,0.000074,0.000227,0.0,0.000769
feature_698,0.903796,0.904774,0.000979,2,0.000029,0.000090,0.0,0.000304
feature_536,0.903796,0.904774,0.000979,2,0.000459,0.001408,0.0,0.004774
feature_987,0.903796,0.904765,0.000969,2,0.000409,0.001255,0.0,0.004256
feature_532,0.903796,0.904774,0.000979,2,0.000284,0.000870,0.0,0.002952
feature_507,0.903796,0.904774,0.000979,2,0.000128,0.000394,0.0,0.001335


Lowest cardinality features:


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,std_train,min_train,max_train
feature_0,0.000000,0.000000,0.000000,1,0.500000,0.000000,0.5,0.500000
feature_1064,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.0,0.000000
feature_49,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.0,0.000000
feature_1057,1.000000,1.000000,0.000000,1,0.000000,0.000000,0.0,0.000000
feature_2,0.000000,0.000000,0.000000,1,1.000000,0.000000,1.0,1.000000
feature_3,0.000000,0.000000,0.000000,1,1.000000,0.000000,1.0,1.000000
feature_1,0.000000,0.000000,0.000000,1,0.500000,0.000000,0.5,0.500000
feature_504,0.903796,0.904774,0.000979,2,0.000003,0.000010,0.0,0.000035
feature_240,0.863259,0.865527,0.002268,2,0.027039,0.067938,0.0,0.197739
feature_246,0.863259,0.865527,0.002268,2,0.024196,0.060794,0.0,0.176945


## 3. Дубликаты, пересечение train/test и честная валидация

В табличных соревнованиях дубликаты строк — частая причина неверной валидации. Если одинаковый набор признаков попал и в train, и в validation, то модель может выглядеть лучше, чем она реально обобщает.

Что проверяем:

- полные дубликаты feature-vector'ов внутри train;
- конфликтные дубликаты: одинаковые признаки, но разные target;
- совпадения feature-vector'ов между train и test.

Если между train и test есть точные совпадения, это можно использовать как аккуратный postprocessing для сабмита: для совпавших test-строк мы уже знаем target из train. Но в отчёте лучше описывать это как анализ дублей, а не как «магическое читерство».

In [6]:
from pandas.util import hash_pandas_object

# Hash по всем feature columns. Коллизии теоретически возможны, но практически крайне маловероятны.
# Для финальной проверки конфликтных групп можно отдельно сравнивать сами строки внутри подозрительных hash-групп.

t0 = time.time()
train_feature_hash = hash_pandas_object(train[feature_cols], index=False).astype("uint64")
test_feature_hash = hash_pandas_object(test[feature_cols], index=False).astype("uint64")
print(f"Hashes computed in {time.time() - t0:.1f}s")

train_dup_counts = train_feature_hash.value_counts()
test_dup_counts = test_feature_hash.value_counts()

print("Train rows participating in duplicated feature vectors:", int(train_dup_counts[train_dup_counts > 1].sum()))
print("Number of duplicated feature-vector groups in train:", int((train_dup_counts > 1).sum()))
print("Test rows participating in duplicated feature vectors:", int(test_dup_counts[test_dup_counts > 1].sum()))
print("Number of duplicated feature-vector groups in test:", int((test_dup_counts > 1).sum()))

train_hash_target_stats = (
    pd.DataFrame({"_hash": train_feature_hash, TARGET_COL: train[TARGET_COL].values})
    .groupby("_hash")[TARGET_COL]
    .agg(["count", "sum", "mean"])
    .sort_values("count", ascending=False)
)

conflict_hashes = train_hash_target_stats[
    (train_hash_target_stats["count"] > 1) &
    (train_hash_target_stats["mean"] > 0) &
    (train_hash_target_stats["mean"] < 1)
]

print("Conflict duplicate groups in train:", len(conflict_hashes))
display(conflict_hashes.head(20))

# Пересечение train/test по feature-vector hash.
train_hash_to_target = (
    pd.DataFrame({"_hash": train_feature_hash, TARGET_COL: train[TARGET_COL].values})
    .groupby("_hash")[TARGET_COL]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

test_overlap = pd.DataFrame({ID_COL: test[ID_COL].values, "_hash": test_feature_hash.values}).merge(
    train_hash_to_target, on="_hash", how="inner"
)

print("Test rows with feature-vector also present in train:", len(test_overlap))
print("Known positive targets among overlapped rows, by train mean sum:", test_overlap["mean"].sum())
display(test_overlap.head())

Hashes computed in 1.0s
Train rows participating in duplicated feature vectors: 18746
Number of duplicated feature-vector groups in train: 7139
Test rows participating in duplicated feature vectors: 4715
Number of duplicated feature-vector groups in test: 1545
Conflict duplicate groups in train: 93


,count,sum,mean
_hash,,,
10810876695524086779,364,6,0.016484
2719412945535468795,316,3,0.009494
15024679413994247915,315,2,0.006349
6518861788221113675,161,3,0.018634
3627506730639429941,108,2,0.018519
9844721106959895500,48,1,0.020833
12907178941986223151,45,2,0.044444
7436395924665408389,44,2,0.045455
16284551266699955007,44,2,0.045455


Test rows with feature-vector also present in train: 8128
Known positive targets among overlapped rows, by train mean sum: 169.41056512390355


,index,_hash,count,sum,mean
0,132875,11246226303984376264,1,0,0.000000
1,280898,13864781300107392152,24,0,0.000000
2,216290,10003429060184137555,1,1,1.000000
3,199917,6339280815843837279,1,0,0.000000
4,289908,2719412945535468795,316,3,0.009494


In [7]:
# Сохраняем потенциальный leakage/postprocess файл.
# Если у test-строки есть точное совпадение в train, можно заменить score на mean target этой группы.
# Для конфликтных групп mean может быть между 0 и 1 — это честнее, чем брать 0/1.

ARTIFACT_DIR = PROJECT_ROOT / "processed_data"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

leakage_candidates = test_overlap[[ID_COL, "mean", "count", "sum"]].rename(
    columns={"mean": "known_target_mean", "count": "train_duplicate_count", "sum": "train_positive_count"}
)
leakage_candidates.to_csv(ARTIFACT_DIR / "test_train_exact_overlap.csv", index=False)
print("Saved:", ARTIFACT_DIR / "test_train_exact_overlap.csv")

Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/test_train_exact_overlap.csv


## 4. Валидация без утечки через дубли

Если в train есть одинаковые строки, желательно, чтобы все одинаковые feature-vector'ы попадали в один и тот же fold. Для этого используем `StratifiedGroupKFold`, где group = hash feature-vector'а.

Это может чуть понизить локальный AUC, зато метрика станет честнее.

In [8]:
y = train[TARGET_COL].astype("int8").values

def make_cv_splits(y, groups=None, n_splits=5, random_state=42):
    if groups is not None and HAS_STRATIFIED_GROUP_KFOLD:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        return list(splitter.split(np.zeros(len(y)), y, groups=groups))
    else:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        return list(splitter.split(np.zeros(len(y)), y))

cv_splits = make_cv_splits(y, groups=train_feature_hash.values, n_splits=N_SPLITS, random_state=RANDOM_STATE)

fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(cv_splits):
    fold_rows.append({
        "fold": fold,
        "train_size": len(tr_idx),
        "valid_size": len(va_idx),
        "valid_target_rate": float(y[va_idx].mean()),
        "valid_pos": int(y[va_idx].sum()),
    })
fold_df = pd.DataFrame(fold_rows)
display(fold_df)

,fold,train_size,valid_size,valid_target_rate,valid_pos
0,0,198376,49596,0.013509,670
1,1,198378,49594,0.013490,669
2,2,198378,49594,0.013490,669
3,3,198378,49594,0.013490,669
4,4,198378,49594,0.013490,669


## 5. Удаление явно плохих признаков

Удаляем только то, что почти наверняка не несёт информации:

- константы;
- почти константы, где одно значение занимает больше `99.9%` строк;
- полностью пустые признаки, если бы они были.

Важно: мы **не удаляем выбросы по z-score**. В таких задачах редкий большой пик может быть самым сильным сигналом.

In [9]:
def get_top_value_share(s: pd.Series) -> float:
    vc = s.value_counts(dropna=False, normalize=True)
    if len(vc) == 0:
        return 1.0
    return float(vc.iloc[0])

# На 1367 колонках это нормально по времени. Если станет медленно, можно заменить на более грубую проверку nunique <= 1 + zero_frac.
t0 = time.time()
top_value_share = train[feature_cols].apply(get_top_value_share)
print(f"Top value shares computed in {time.time() - t0:.1f}s")

feature_profile = basic_profile.copy()
feature_profile["top_value_share_train"] = top_value_share
feature_profile["is_constant"] = feature_profile["nunique_train"] <= 1
feature_profile["is_almost_constant"] = feature_profile["top_value_share_train"] >= LOW_VARIANCE_TOP_SHARE_THRESHOLD

low_variance_cols = feature_profile.index[
    feature_profile["is_constant"] | feature_profile["is_almost_constant"]
].tolist()

print("Low-variance columns to drop:", len(low_variance_cols))
print(low_variance_cols[:100])

good_base_cols = [c for c in feature_cols if c not in low_variance_cols]
print("Good base columns:", len(good_base_cols))

Top value shares computed in 1.5s
Low-variance columns to drop: 7
['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_49', 'feature_1057', 'feature_1064']
Good base columns: 1360


## 6. Полный feature summary: корреляции, drift, ранжирование

Считаем таблицу `feature_profile_full.csv`. Она нужна не только для EDA, но и для автоматического выбора feature sets.

Что означают поля:

- `abs_pearson_target` — линейная связь с target. Малое значение не значит, что признак бесполезен: бустинг может ловить нелинейности.
- `abs_spearman_target_sample` — монотонная связь на стратифицированной выборке.
- `drift_z_mean` — насколько среднее в test отличается от train в единицах стандартного отклонения.
- `zero_frac_delta` — насколько отличается доля нулей в train/test.

Сильный drift не всегда значит «удалить признак». Иногда drift отражает реальный сдвиг test, и удаление ухудшит качество. Поэтому мы сначала только флагируем.

In [10]:
def make_stratified_sample_indices(y: np.ndarray, max_size: int, random_state: int = 42) -> np.ndarray:
    if len(y) <= max_size:
        return np.arange(len(y))
    idx = np.arange(len(y))
    _, sample_idx = train_test_split(
        idx,
        test_size=max_size,
        stratify=y,
        random_state=random_state,
    )
    return np.sort(sample_idx)

feature_profile = feature_profile.loc[good_base_cols].copy()

# Pearson на полном train.
print("Computing Pearson correlations...")
feature_profile["abs_pearson_target"] = train[good_base_cols].corrwith(train[TARGET_COL]).abs().fillna(0)

# Spearman на выборке, потому что полный Spearman по 1360 колонкам может быть ощутимо медленным.
print("Computing sample Spearman correlations...")
sample_idx = make_stratified_sample_indices(y, SPEARMAN_SAMPLE_SIZE, RANDOM_STATE)
feature_profile["abs_spearman_target_sample"] = (
    train.iloc[sample_idx][good_base_cols]
    .corrwith(train.iloc[sample_idx][TARGET_COL], method="spearman")
    .abs()
    .fillna(0)
)

# Drift между train/test по среднему и доле нулей.
mean_train = train[good_base_cols].mean()
mean_test = test[good_base_cols].mean()
std_train = train[good_base_cols].std().replace(0, np.nan)
feature_profile["mean_test"] = mean_test
feature_profile["drift_z_mean"] = ((mean_test - mean_train).abs() / std_train).fillna(0)
feature_profile["zero_frac_delta"] = (test[good_base_cols].eq(0).mean() - train[good_base_cols].eq(0).mean()).abs()
feature_profile["is_drift_flag"] = feature_profile["drift_z_mean"] >= DRIFT_Z_THRESHOLD_TO_FLAG

feature_profile = feature_profile.sort_values("abs_spearman_target_sample", ascending=False)
feature_profile.to_csv(ARTIFACT_DIR / "feature_profile_full.csv")

print("Saved:", ARTIFACT_DIR / "feature_profile_full.csv")
print("Top by Spearman:")
display(feature_profile.head(30))

print("Top drift features:")
display(feature_profile.sort_values("drift_z_mean", ascending=False).head(20))

Computing Pearson correlations...
Computing sample Spearman correlations...
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/feature_profile_full.csv
Top by Spearman:


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,std_train,min_train,max_train,top_value_share_train,is_constant,is_almost_constant,abs_pearson_target,abs_spearman_target_sample,mean_test,drift_z_mean,is_drift_flag
feature_72,0.321577,0.321800,0.000224,4975,206.648727,1036.980835,0.000000,1.956190e+05,0.321577,False,False,0.009419,0.020060,206.698578,0.000048,False
feature_363,0.442159,0.440465,0.001694,5365,206.058044,2000.254272,0.000000,2.700170e+05,0.442159,False,False,0.007410,0.020032,216.044952,0.004993,False
feature_282,0.290299,0.289441,0.000858,5262,233.988739,1113.954224,0.000000,1.997700e+05,0.290299,False,False,0.009695,0.019656,234.384216,0.000355,False
feature_1094,0.000000,0.000000,0.000000,203027,0.267927,0.169308,0.000467,9.827985e-01,0.153082,False,False,0.020181,0.019305,0.267716,0.001249,False
feature_283,0.257114,0.255669,0.001444,5644,268.968964,1210.511841,0.000000,2.274120e+05,0.257114,False,False,0.010140,0.018988,269.937561,0.000800,False
feature_366,0.402348,0.400597,0.001751,1543,34.725082,138.086472,0.000000,1.585400e+04,0.402348,False,False,0.011670,0.018838,35.203197,0.003462,False
feature_41,0.000000,0.000000,0.000000,174004,0.765057,0.133459,0.116091,1.327086e+00,0.130269,False,False,0.013425,0.018627,0.765291,0.001755,False
feature_379,0.663821,0.664405,0.000584,1279,15.243561,183.665131,0.000000,4.032400e+04,0.663821,False,False,0.003972,0.018432,15.909376,0.003625,False
feature_93,0.679202,0.678962,0.000240,1496,18.109451,228.948853,0.000000,2.984700e+04,0.679202,False,False,0.005379,0.017760,19.901491,0.007827,False
feature_377,0.144004,0.141888,0.002116,8905,561.302979,2643.070557,0.000000,2.129550e+05,0.144004,False,False,0.011241,0.017671,586.779846,0.009639,False


Top drift features:


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,std_train,min_train,max_train,top_value_share_train,is_constant,is_almost_constant,abs_pearson_target,abs_spearman_target_sample,mean_test,drift_z_mean,is_drift_flag
feature_1007,0.903796,0.904765,0.000969,2,0.000057,0.000176,0.0,0.000597,0.903796,False,False,0.000581,0.000828,0.000763,4.004632,False
feature_835,0.903788,0.904727,0.000940,4,0.000178,0.016565,0.0,8.000000,0.903788,False,False,0.000299,0.000833,0.025373,1.520998,False
feature_1061,0.903796,0.904756,0.000960,2,0.000012,0.000036,0.0,0.000124,0.903796,False,False,0.000581,0.000828,0.000049,1.029635,False
feature_587,0.903788,0.904765,0.000978,3,0.000812,0.003758,0.0,1.000000,0.903788,False,False,0.000632,0.000833,0.002480,0.443826,False
feature_702,0.903792,0.904765,0.000973,3,0.002568,0.052750,0.0,26.000000,0.903792,False,False,0.000316,0.000828,0.024758,0.420655,False
feature_509,0.903755,0.904746,0.000991,9,0.007017,0.168367,0.0,72.000000,0.903755,False,False,0.000518,0.000844,0.076329,0.411676,False
feature_674,0.903425,0.904426,0.001002,70,0.757284,130.373535,0.0,58186.000000,0.903425,False,False,0.000490,0.000962,30.765835,0.230174,False
feature_866,0.903763,0.904746,0.000983,9,0.003878,0.504398,0.0,156.000000,0.903763,False,False,0.000527,0.000855,0.086951,0.164698,False
feature_659,0.903723,0.904680,0.000957,18,0.042810,8.357028,0.0,4130.000000,0.903723,False,False,0.000331,0.000898,0.939597,0.107309,False
feature_1015,0.903667,0.904643,0.000976,31,0.064829,11.854844,0.0,3409.000000,0.903667,False,False,0.000416,0.000882,1.206850,0.096334,False


## 7. Mutual information

Корреляция ловит только простые связи. Mutual information пытается поймать более общий сигнал: насколько знание признака уменьшает неопределённость target.

Минусы:

- может быть шумной при редком target;
- на больших данных считается медленнее;
- значения не стоит интерпретировать как «проценты важности».

Используем её как ещё один голос в общем ранжировании, а не как единственный критерий.

In [11]:
if RUN_MUTUAL_INFO:
    print("Computing mutual information on a stratified sample...")
    mi_idx = make_stratified_sample_indices(y, MI_SAMPLE_SIZE, RANDOM_STATE)
    X_mi = train.iloc[mi_idx][good_base_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_mi = train.iloc[mi_idx][TARGET_COL].values

    t0 = time.time()
    mi_values = mutual_info_classif(
        X_mi,
        y_mi,
        discrete_features=False,
        random_state=RANDOM_STATE,
        n_neighbors=3,
    )
    print(f"MI computed in {time.time() - t0:.1f}s")

    feature_profile["mutual_info_sample"] = pd.Series(mi_values, index=good_base_cols)
else:
    feature_profile["mutual_info_sample"] = 0.0

feature_profile["mutual_info_sample"] = feature_profile["mutual_info_sample"].fillna(0)
feature_profile.sort_values("mutual_info_sample", ascending=False).head(30)

Computing mutual information on a stratified sample...
MI computed in 153.1s


,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,std_train,min_train,max_train,top_value_share_train,is_constant,is_almost_constant,abs_pearson_target,abs_spearman_target_sample,mean_test,drift_z_mean,is_drift_flag,mutual_info_sample
feature_1140,0.434214,0.429559,0.004655,2,0.565786,0.495654,0.000000,1.000000,0.565786,False,False,0.007977,0.003601,0.570441,0.009391,False,0.013632
feature_1098,0.009719,0.009805,0.000086,3,0.711475,0.695763,-1.000000,1.000000,0.850878,False,False,0.001839,0.001196,0.715208,0.005364,False,0.013507
feature_1082,0.006222,0.006427,0.000204,3,0.714972,0.694690,-1.000000,1.000000,0.854375,False,False,0.001625,0.001583,0.718586,0.005202,False,0.013025
feature_1168,0.496197,0.495069,0.001128,2,0.503803,0.499987,0.000000,1.000000,0.503803,False,False,0.003127,0.001147,0.504931,0.002256,False,0.012572
feature_1138,0.513122,0.509758,0.003365,2,0.486878,0.499829,0.000000,1.000000,0.513122,False,False,0.008608,0.003650,0.490242,0.006732,False,0.012081
feature_1113,0.000000,0.000000,0.000000,3,18.777813,12.135384,-1.000000,27.000000,0.568516,False,False,0.003795,0.004167,18.895439,0.009693,False,0.011296
feature_1066,0.000000,0.000000,0.000000,60579,0.006379,0.004886,0.000084,0.321850,0.748484,False,False,0.000488,0.000384,0.006357,0.004362,False,0.010522
feature_1070,0.000000,0.000000,0.000000,82089,0.006179,0.005761,0.000052,0.344588,0.658804,False,False,0.000908,0.005570,0.006182,0.000484,False,0.010393
feature_1124,0.000000,0.000000,0.000000,3,26.108557,16.626471,-1.000000,37.000000,0.508457,False,False,0.004026,0.006545,26.253721,0.008731,False,0.009588
feature_1180,0.563213,0.561238,0.001975,2,0.436787,0.495989,0.000000,1.000000,0.563213,False,False,0.007505,0.003366,0.438762,0.003982,False,0.009381


## 8. LightGBM importance по фолдам

Для табличных данных с нелинейностями LightGBM importance обычно полезнее простой корреляции. Мы обучаем несколько моделей на CV-фолдах и агрегируем важности.

Смотрим два типа importance:

- `gain`: сколько модель выиграла в лоссе от сплитов по признаку;
- `split`: сколько раз признак использовался в деревьях.

Более устойчивые признаки — те, которые получают важность на разных фолдах.

In [12]:
def train_lgb_importance(
    train_df: pd.DataFrame,
    y: np.ndarray,
    features: List[str],
    cv_splits: List[Tuple[np.ndarray, np.ndarray]],
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not HAS_LIGHTGBM:
        raise ImportError("lightgbm is not installed. Install with: pip install lightgbm")

    rows = []
    val_rows = []

    scale_pos_weight = (len(y) - y.sum()) / max(y.sum(), 1)
    base_params = dict(
        objective="binary",
        metric="auc",
        boosting_type="gbdt",
        learning_rate=0.03,
        num_leaves=63,
        min_data_in_leaf=80,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        bagging_freq=1,
        lambda_l1=0.1,
        lambda_l2=10.0,
        scale_pos_weight=scale_pos_weight,
        verbosity=-1,
        seed=random_state,
        feature_pre_filter=False,
        num_threads=max(os.cpu_count() - 1, 1),
    )

    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"Fold {fold}: train={len(tr_idx)}, valid={len(va_idx)}")
        X_tr = train_df.iloc[tr_idx][features]
        X_va = train_df.iloc[va_idx][features]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=True)
        dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=True)

        model = lgb.train(
            base_params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            valid_names=["valid"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=150, verbose=False),
                lgb.log_evaluation(period=200),
            ],
        )

        pred = model.predict(X_va, num_iteration=model.best_iteration)
        auc = roc_auc_score(y_va, pred)
        val_rows.append({"fold": fold, "auc": auc, "best_iteration": model.best_iteration})
        print(f"Fold {fold} AUC: {auc:.6f}, best_iteration={model.best_iteration}")

        for imp_type in ["gain", "split"]:
            imp = model.feature_importance(importance_type=imp_type)
            for f, v in zip(features, imp):
                rows.append({"fold": fold, "feature": f, "importance_type": imp_type, "importance": float(v)})

        del X_tr, X_va, dtrain, dvalid, model
        gc.collect()

    imp_raw = pd.DataFrame(rows)
    val_df = pd.DataFrame(val_rows)
    return imp_raw, val_df

if RUN_LGB_IMPORTANCE:
    lgb_imp_raw, lgb_val_df = train_lgb_importance(train, y, good_base_cols, cv_splits, RANDOM_STATE)

    lgb_imp_agg = (
        lgb_imp_raw
        .pivot_table(index="feature", columns="importance_type", values="importance", aggfunc=["mean", "std"])
    )
    lgb_imp_agg.columns = [f"lgb_{stat}_{imp_type}" for stat, imp_type in lgb_imp_agg.columns]
    lgb_imp_agg = lgb_imp_agg.fillna(0).reset_index()

    # Fold presence: на скольких фолдах gain > 0.
    gain_presence = (
        lgb_imp_raw[lgb_imp_raw["importance_type"] == "gain"]
        .assign(nonzero=lambda d: d["importance"] > 0)
        .groupby("feature")["nonzero"]
        .sum()
        .rename("lgb_gain_nonzero_folds")
    )
    lgb_imp_agg = lgb_imp_agg.merge(gain_presence, on="feature", how="left").fillna({"lgb_gain_nonzero_folds": 0})

    lgb_imp_agg = lgb_imp_agg.sort_values("lgb_mean_gain", ascending=False)

    lgb_imp_raw.to_csv(ARTIFACT_DIR / "lgb_importance_raw_by_fold.csv", index=False)
    lgb_imp_agg.to_csv(ARTIFACT_DIR / "lgb_importance_aggregated.csv", index=False)
    lgb_val_df.to_csv(ARTIFACT_DIR / "lgb_importance_cv_scores.csv", index=False)

    print("CV AUC:")
    display(lgb_val_df)
    print("Top LGB features:")
    display(lgb_imp_agg.head(40))
else:
    lgb_imp_agg = pd.DataFrame({"feature": good_base_cols, "lgb_mean_gain": 0.0, "lgb_mean_split": 0.0, "lgb_gain_nonzero_folds": 0})
    lgb_val_df = pd.DataFrame()

Fold 0: train=198376, valid=49596
[200]	valid's auc: 0.622469
Fold 0 AUC: 0.624954, best_iteration=127
Fold 1: train=198378, valid=49594
[200]	valid's auc: 0.639191
Fold 1 AUC: 0.639461, best_iteration=208
Fold 2: train=198378, valid=49594
[200]	valid's auc: 0.625263
Fold 2 AUC: 0.626546, best_iteration=152
Fold 3: train=198378, valid=49594
[200]	valid's auc: 0.640244
Fold 3 AUC: 0.642195, best_iteration=169
Fold 4: train=198378, valid=49594
[200]	valid's auc: 0.638085
Fold 4 AUC: 0.639838, best_iteration=233
CV AUC:


,fold,auc,best_iteration
0,0,0.624954,127
1,1,0.639461,208
2,2,0.626546,152
3,3,0.642195,169
4,4,0.639838,233


Top LGB features:


,feature,lgb_mean_gain,lgb_mean_split,lgb_std_gain,lgb_std_split,lgb_gain_nonzero_folds
707,feature_41,53825.488260,99.4,8285.334355,20.586403,5
729,feature_43,50978.666940,100.8,6279.245072,19.136353,5
103,feature_1094,43217.805350,100.6,2283.552367,15.009997,5
670,feature_377,40708.784912,42.4,35770.812585,13.501852,5
135,feature_1122,36904.444437,102.4,6233.752159,33.649666,5
95,feature_1087,36526.582431,69.8,18745.092144,18.390215,5
100,feature_1091,35244.175848,98.2,10817.736219,33.907226,5
331,feature_13,34813.523442,71.6,6362.595327,20.305172,5
497,feature_22,32657.650566,89.8,6852.891365,29.320641,5
431,feature_16,32292.380182,90.0,3493.222782,13.601471,5


## 9. Общее ранжирование признаков

Теперь объединяем несколько источников сигнала в один рейтинг:

- корреляция;
- Spearman;
- mutual information;
- LightGBM gain;
- устойчивость по фолдам;
- штраф за сильный train/test drift.

Это не математически «единственно правильная формула». Это практический ранкер, чтобы собрать разумные `top300/top500/top800` feature sets.

In [13]:
def minmax_rank_score(s: pd.Series, higher_is_better: bool = True) -> pd.Series:
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0)
    if s.nunique() <= 1:
        return pd.Series(0.0, index=s.index)
    rank = s.rank(method="average", ascending=not higher_is_better)
    # Лучший получает почти 1, худший почти 0.
    return 1.0 - (rank - 1) / max(len(rank) - 1, 1)

ranking = feature_profile.reset_index().rename(columns={"index": "feature"})
ranking = ranking.merge(lgb_imp_agg, on="feature", how="left")
ranking = ranking.fillna(0)

ranking["score_corr"] = minmax_rank_score(ranking["abs_pearson_target"])
ranking["score_spearman"] = minmax_rank_score(ranking["abs_spearman_target_sample"])
ranking["score_mi"] = minmax_rank_score(ranking["mutual_info_sample"])
ranking["score_lgb_gain"] = minmax_rank_score(ranking.get("lgb_mean_gain", pd.Series(0, index=ranking.index)))
ranking["score_lgb_stability"] = ranking.get("lgb_gain_nonzero_folds", pd.Series(0, index=ranking.index)) / max(N_SPLITS, 1)
ranking["score_drift_penalty"] = minmax_rank_score(ranking["drift_z_mean"], higher_is_better=False)

ranking["combined_score"] = (
    0.15 * ranking["score_corr"] +
    0.15 * ranking["score_spearman"] +
    0.15 * ranking["score_mi"] +
    0.35 * ranking["score_lgb_gain"] +
    0.15 * ranking["score_lgb_stability"] +
    0.05 * ranking["score_drift_penalty"]
)

ranking = ranking.sort_values("combined_score", ascending=False).reset_index(drop=True)
ranking.to_csv(ARTIFACT_DIR / "feature_ranking_combined.csv", index=False)

print("Saved:", ARTIFACT_DIR / "feature_ranking_combined.csv")
display(ranking.head(50))

Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/feature_ranking_combined.csv


,feature,zero_frac_train,zero_frac_test,zero_frac_delta,nunique_train,mean_train,std_train,min_train,max_train,top_value_share_train,is_constant,is_almost_constant,abs_pearson_target,abs_spearman_target_sample,mean_test,drift_z_mean,is_drift_flag,mutual_info_sample,lgb_mean_gain,lgb_mean_split,lgb_std_gain,lgb_std_split,lgb_gain_nonzero_folds,score_corr,score_spearman,score_mi,score_lgb_gain,score_lgb_stability,score_drift_penalty,combined_score
0,feature_1110,0.000000,0.000000,0.000000,7,4.789210,1.646821,-1.000000,7.000000e+00,0.325271,False,False,0.013611,0.016506,4.788782,0.000260,False,0.004769,10697.415823,23.6,2898.870652,8.384510,5,0.991906,0.988962,0.972038,0.914643,1.0,0.949227,0.960522
1,feature_16,0.000000,0.000000,0.000000,19748,0.649797,0.314333,0.002809,3.521739e+00,0.122332,False,False,0.008008,0.011260,0.650118,0.001020,False,0.001292,32292.380182,90.0,3493.222782,13.601471,5,0.944812,0.934511,0.885210,0.993377,1.0,0.837380,0.954231
2,feature_34,0.000000,0.000000,0.000000,9082,0.498150,0.334547,0.005607,3.600000e+00,0.122332,False,False,0.012572,0.008287,0.498116,0.000102,False,0.000984,28107.083409,68.0,5382.070517,20.700242,5,0.988227,0.860927,0.825607,0.985283,1.0,0.982340,0.945180
3,feature_41,0.000000,0.000000,0.000000,174004,0.765057,0.133459,0.116091,1.327086e+00,0.130269,False,False,0.013425,0.018627,0.765291,0.001755,False,0.000639,53825.488260,99.4,8285.334355,20.586403,5,0.989698,0.995585,0.728477,1.000000,1.0,0.720383,0.943083
4,feature_1094,0.000000,0.000000,0.000000,203027,0.267927,0.169308,0.000467,9.827985e-01,0.153082,False,False,0.020181,0.019305,0.267716,0.001249,False,0.000516,43217.805350,100.6,2283.552367,15.009997,5,1.000000,0.997792,0.685063,0.998528,1.0,0.794702,0.941648
5,feature_1202,0.422939,0.424864,0.001925,3,0.277495,0.707149,-1.000000,1.000000e+00,0.427278,False,False,0.012036,0.013543,0.280332,0.004012,False,0.005263,15772.951288,4.6,12908.390558,2.880972,5,0.986755,0.962472,0.974246,0.944077,1.0,0.383370,0.938116
6,feature_43,0.000000,0.000000,0.000000,177664,0.502538,0.754663,0.007585,6.496452e+00,0.130269,False,False,0.011912,0.012133,0.506474,0.005216,False,0.001306,50978.666940,100.8,6279.245072,19.136353,5,0.986019,0.949227,0.888889,0.999264,1.0,0.292127,0.937969
7,feature_1087,0.000000,0.000000,0.000000,201112,0.147538,0.083700,0.000573,9.269438e-01,0.160268,False,False,0.015832,0.010554,0.147443,0.001129,False,0.000655,36526.582431,69.8,18745.092144,18.390215,5,0.995585,0.915379,0.735099,0.996321,1.0,0.817513,0.936497
8,feature_1122,0.000000,0.000000,0.000000,2788,0.391608,0.105309,0.050000,7.600000e-01,0.041908,False,False,0.015188,0.016977,0.391526,0.000780,False,0.000416,36904.444437,102.4,6233.752159,33.649666,5,0.994113,0.990434,0.630611,0.997057,1.0,0.866814,0.934584
9,feature_29,0.000000,0.000000,0.000000,172732,0.896781,0.077598,0.511350,1.605458e+00,0.130269,False,False,0.009696,0.008991,0.896802,0.000273,False,0.000689,29524.897641,75.6,3974.887507,18.022209,5,0.971302,0.885210,0.744665,0.986755,1.0,0.947756,0.932929


## 10. Корреляционное прореживание топовых признаков

Если два признака почти одинаковые, можно оставить один. Это полезно для линейных моделей, PCA, target encoding и ускорения. Для бустинга это не всегда обязательно, но уменьшает шум.

Важно: делаем pruning только внутри топовых признаков, а не по всем 1360 колонкам, чтобы не тратить много времени на огромную корреляционную матрицу.

In [14]:
def correlation_prune_features(
    df: pd.DataFrame,
    features: List[str],
    max_features_to_check: int = 800,
    threshold: float = 0.995,
) -> List[str]:
    selected = features[:max_features_to_check]
    if len(selected) <= 1:
        return selected
    corr = df[selected].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = set()
    for col in upper.columns:
        if (upper[col] > threshold).any():
            to_drop.add(col)
    pruned = [f for f in selected if f not in to_drop]
    # Остальные признаки после max_features_to_check просто добавляем как есть.
    pruned += features[max_features_to_check:]
    return pruned

ranked_features = ranking["feature"].tolist()
ranked_features_pruned = correlation_prune_features(train, ranked_features, max_features_to_check=min(800, len(ranked_features)), threshold=0.995)

print("Ranked features before pruning:", len(ranked_features))
print("Ranked features after pruning:", len(ranked_features_pruned))
print("Dropped in first 800:", len(set(ranked_features[:800]) - set(ranked_features_pruned[:800])))

Ranked features before pruning: 1360
Ranked features after pruning: 1349
Dropped in first 800: 11


## 11. Feature engineering: row-level sparse statistics

Большая доля нулей — это не просто техническая особенность, а потенциальный сигнал. Например, «пользователь активен во многих поведенческих группах» может быть важнее любого одного признака.

Добавляем признаки по строке:

- количество ненулевых признаков;
- доля нулей;
- количество положительных/отрицательных значений;
- сумма, сумма модулей, среднее, std, max, min;
- квантили;
- L2-норма;
- `log1p` от сумм, чтобы огромные значения не доминировали.

In [15]:
def add_row_stats(df: pd.DataFrame, cols: List[str], prefix: str) -> pd.DataFrame:
    X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    arr = X.to_numpy(dtype=np.float32, copy=False)

    out = pd.DataFrame(index=df.index)
    zero_mask = arr == 0
    pos_mask = arr > 0
    neg_mask = arr < 0

    out[f"{prefix}_nonzero_count"] = (~zero_mask).sum(axis=1).astype("int16")
    out[f"{prefix}_zero_frac"] = zero_mask.mean(axis=1).astype("float32")
    out[f"{prefix}_positive_count"] = pos_mask.sum(axis=1).astype("int16")
    out[f"{prefix}_negative_count"] = neg_mask.sum(axis=1).astype("int16")

    row_sum = arr.sum(axis=1)
    abs_sum = np.abs(arr).sum(axis=1)
    out[f"{prefix}_sum"] = row_sum.astype("float32")
    out[f"{prefix}_abs_sum"] = abs_sum.astype("float32")
    out[f"{prefix}_log1p_abs_sum"] = np.log1p(abs_sum).astype("float32")
    out[f"{prefix}_mean"] = arr.mean(axis=1).astype("float32")
    out[f"{prefix}_std"] = arr.std(axis=1).astype("float32")
    out[f"{prefix}_min"] = arr.min(axis=1).astype("float32")
    out[f"{prefix}_max"] = arr.max(axis=1).astype("float32")
    out[f"{prefix}_q25"] = np.quantile(arr, 0.25, axis=1).astype("float32")
    out[f"{prefix}_q50"] = np.quantile(arr, 0.50, axis=1).astype("float32")
    out[f"{prefix}_q75"] = np.quantile(arr, 0.75, axis=1).astype("float32")
    out[f"{prefix}_l2_norm"] = np.sqrt((arr ** 2).sum(axis=1)).astype("float32")

    return out

print("Building global row stats...")
train_meta = add_row_stats(train, good_base_cols, "all")
test_meta = add_row_stats(test, good_base_cols, "all")
print(train_meta.shape, test_meta.shape)
display(train_meta.head())

Building global row stats...
(247972, 15) (106274, 15)


,all_nonzero_count,all_zero_frac,all_positive_count,all_negative_count,all_sum,all_abs_sum,all_log1p_abs_sum,all_mean,all_std,all_min,all_max,all_q25,all_q50,all_q75,all_l2_norm
0,1124,0.173529,1104,20,95519.625000,95559.625000,11.467516,70.235016,665.595337,-1.0,13230.237305,0.011121,0.252916,1.810390,24682.248047
1,1282,0.057353,1103,179,95358.242188,95716.242188,11.469153,70.116356,665.609619,-1.0,13230.237305,0.009211,0.255198,1.722785,24682.248047
2,264,0.805882,263,1,351827.031250,351829.031250,12.770904,258.696350,3116.980469,-1.0,53508.000000,0.000000,0.000000,0.000000,115344.515625
3,167,0.877206,161,6,54329.992188,54341.992188,10.903071,39.948524,499.825073,-1.0,8144.000000,0.000000,0.000000,0.000000,18491.375000
4,278,0.795588,277,1,437379.562500,437381.562500,12.988564,321.602631,4518.102051,-1.0,70242.000000,0.000000,0.000000,0.000000,167039.921875


## 12. Block statistics

Признаки называются `feature_0`, `feature_1`, ..., но часто в таких датасетах соседние колонки могут относиться к похожим источникам. Мы не знаем семантику, но можем добавить агрегаты по блокам колонок.

Например, для блоков по 100 признаков считаем: сколько ненулевых, сумма, максимум, std. Это дешёвый способ дать модели информацию о «сегментах активности».

In [16]:
def make_index_blocks(cols: List[str], block_size: int = 100) -> Dict[str, List[str]]:
    def feature_num(c: str) -> int:
        return int(c.split("_")[1])
    cols_sorted = sorted(cols, key=feature_num)
    blocks = {}
    for start in range(0, len(cols_sorted), block_size):
        block_cols = cols_sorted[start:start + block_size]
        if block_cols:
            blocks[f"block_{start:04d}_{start + len(block_cols) - 1:04d}"] = block_cols
    return blocks


def add_light_block_stats(df: pd.DataFrame, blocks: Dict[str, List[str]]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for name, cols in blocks.items():
        X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
        arr = X.to_numpy(dtype=np.float32, copy=False)
        out[f"{name}_nonzero_count"] = (arr != 0).sum(axis=1).astype("int16")
        out[f"{name}_zero_frac"] = (arr == 0).mean(axis=1).astype("float32")
        abs_sum = np.abs(arr).sum(axis=1)
        out[f"{name}_log1p_abs_sum"] = np.log1p(abs_sum).astype("float32")
        out[f"{name}_max"] = arr.max(axis=1).astype("float32")
        out[f"{name}_std"] = arr.std(axis=1).astype("float32")
    return out

blocks = make_index_blocks(good_base_cols, block_size=100)
print("Number of blocks:", len(blocks))

train_block_meta = add_light_block_stats(train, blocks)
test_block_meta = add_light_block_stats(test, blocks)
print(train_block_meta.shape, test_block_meta.shape)
display(train_block_meta.head())

Number of blocks: 14
(247972, 70) (106274, 70)


,block_0000_0099_nonzero_count,block_0000_0099_zero_frac,block_0000_0099_log1p_abs_sum,block_0000_0099_max,block_0000_0099_std,block_0100_0199_nonzero_count,block_0100_0199_zero_frac,block_0100_0199_log1p_abs_sum,block_0100_0199_max,block_0100_0199_std,block_0200_0299_nonzero_count,block_0200_0299_zero_frac,block_0200_0299_log1p_abs_sum,block_0200_0299_max,block_0200_0299_std,block_0300_0399_nonzero_count,block_0300_0399_zero_frac,block_0300_0399_log1p_abs_sum,block_0300_0399_max,block_0300_0399_std,block_0400_0499_nonzero_count,block_0400_0499_zero_frac,block_0400_0499_log1p_abs_sum,block_0400_0499_max,block_0400_0499_std,block_0500_0599_nonzero_count,block_0500_0599_zero_frac,block_0500_0599_log1p_abs_sum,block_0500_0599_max,block_0500_0599_std,block_0600_0699_nonzero_count,block_0600_0699_zero_frac,block_0600_0699_log1p_abs_sum,block_0600_0699_max,block_0600_0699_std,block_0700_0799_nonzero_count,block_0700_0799_zero_frac,block_0700_0799_log1p_abs_sum,block_0700_0799_max,block_0700_0799_std,block_0800_0899_nonzero_count,block_0800_0899_zero_frac,block_0800_0899_log1p_abs_sum,block_0800_0899_max,block_0800_0899_std,block_0900_0999_nonzero_count,block_0900_0999_zero_frac,block_0900_0999_log1p_abs_sum,block_0900_0999_max,block_0900_0999_std,block_1000_1099_nonzero_count,block_1000_1099_zero_frac,block_1000_1099_log1p_abs_sum,block_1000_1099_max,block_1000_1099_std,block_1100_1199_nonzero_count,block_1100_1199_zero_frac,block_1100_1199_log1p_abs_sum,block_1100_1199_max,block_1100_1199_std,block_1200_1299_nonzero_count,block_1200_1299_zero_frac,block_1200_1299_log1p_abs_sum,block_1200_1299_max,block_1200_1299_std,block_1300_1359_nonzero_count,block_1300_1359_zero_frac,block_1300_1359_log1p_abs_sum,block_1300_1359_max,block_1300_1359_std
0,96,0.04,5.889419,49.332207,8.713109,100,0.00,7.326706,784.520752,92.580345,100,0.00,5.794333,56.324829,9.468415,100,0.00,7.499880,471.928802,52.102371,100,0.00,6.660453,159.094818,22.100138,100,0.00,9.476153,5071.001953,652.406677,100,0.00,6.967532,409.535065,46.792557,100,0.00,9.357634,2498.236328,366.160278,100,0.00,10.873399,13230.237305,2134.547852,100,0.0,7.661899,1021.857361,114.202621,100,0.00,9.227264,7695.075195,792.908875,20,0.80,3.033235,2.0,0.471342,2,0.98,1.098612,1.0,0.140000,6,0.900000,1.945910,1.0,0.300000
1,96,0.04,5.889419,49.332207,8.713109,100,0.00,7.326706,784.520752,92.580345,100,0.00,5.794333,56.324829,9.468415,100,0.00,7.499880,471.928802,52.102371,100,0.00,6.660453,159.094818,22.100138,100,0.00,9.476153,5071.001953,652.406677,100,0.00,6.967532,409.535065,46.792557,100,0.00,9.357634,2498.236328,366.160278,100,0.00,10.873399,13230.237305,2134.547852,100,0.0,7.661899,1021.857361,114.202621,97,0.03,9.227002,7695.075195,792.899597,31,0.69,3.402476,1.0,0.482475,100,0.00,4.615120,-1.0,0.000000,58,0.033333,4.077538,1.0,0.571548
2,57,0.43,8.000989,1143.000000,149.645081,18,0.82,9.408535,11243.000000,1118.392822,16,0.84,6.895683,324.000000,47.336613,45,0.55,9.416297,4157.000000,531.438538,14,0.86,7.806696,770.000000,115.223518,3,0.97,10.877009,37921.000000,3951.240479,0,1.00,0.000000,0.000000,0.000000,5,0.95,10.848560,39181.000000,4029.861572,8,0.92,12.046807,53508.000000,9038.786133,0,1.0,0.000000,0.000000,0.000000,40,0.60,10.534195,37572.000000,3738.359619,48,0.52,9.034019,5514.0,558.148987,10,0.90,2.397895,1.0,0.300000,0,1.000000,0.000000,0.0,0.000000
3,47,0.53,3.984665,9.000000,1.318279,6,0.94,4.394449,39.000000,4.597826,5,0.95,5.056246,146.000000,14.528125,12,0.88,6.773080,436.000000,52.372887,3,0.97,3.178054,13.000000,1.468707,2,0.98,8.896998,6983.000000,695.229980,1,0.99,6.700731,812.000000,80.792999,3,0.97,8.918516,6937.000000,691.386536,11,0.89,10.179679,8144.000000,1350.052734,0,1.0,0.000000,0.000000,0.000000,42,0.58,8.509012,4905.000000,488.009064,26,0.74,8.738890,5511.0,548.410278,9,0.91,2.302585,1.0,0.286182,0,1.000000,0.000000,0.0,0.000000
4,50,0.50,5.683479,150.000000,16.540394,6,0.94,6.514713,630.000000,62.714855,7,0.93,3.610918,18.000000,1.931426,19,0.81,6.699501

## 13. Top-k group statistics

Помимо блоков по номеру колонки, считаем агрегаты по самым важным признакам: top-25, top-50, top-100, top-300.

Интуиция: если много сильных признаков одновременно ненулевые или имеют большую сумму, это может быть отдельным сигналом.

In [17]:
def add_topk_group_stats(df: pd.DataFrame, ranked_cols: List[str], ks=(25, 50, 100, 300), prefix="top") -> pd.DataFrame:
    out_parts = []
    for k in ks:
        cols = [c for c in ranked_cols[:k] if c in df.columns]
        if len(cols) == 0:
            continue
        out_parts.append(add_row_stats(df, cols, f"{prefix}{k}"))
    return pd.concat(out_parts, axis=1) if out_parts else pd.DataFrame(index=df.index)

train_topk_meta = add_topk_group_stats(train, ranked_features_pruned, ks=(25, 50, 100, 300), prefix="rank")
test_topk_meta = add_topk_group_stats(test, ranked_features_pruned, ks=(25, 50, 100, 300), prefix="rank")

print(train_topk_meta.shape, test_topk_meta.shape)
display(train_topk_meta.head())

(247972, 60) (106274, 60)


,rank25_nonzero_count,rank25_zero_frac,rank25_positive_count,rank25_negative_count,rank25_sum,rank25_abs_sum,rank25_log1p_abs_sum,rank25_mean,rank25_std,rank25_min,rank25_max,rank25_q25,rank25_q50,rank25_q75,rank25_l2_norm,rank50_nonzero_count,rank50_zero_frac,rank50_positive_count,rank50_negative_count,rank50_sum,rank50_abs_sum,rank50_log1p_abs_sum,rank50_mean,rank50_std,rank50_min,rank50_max,rank50_q25,rank50_q50,rank50_q75,rank50_l2_norm,rank100_nonzero_count,rank100_zero_frac,rank100_positive_count,rank100_negative_count,rank100_sum,rank100_abs_sum,rank100_log1p_abs_sum,rank100_mean,rank100_std,rank100_min,rank100_max,rank100_q25,rank100_q50,rank100_q75,rank100_l2_norm,rank300_nonzero_count,rank300_zero_frac,rank300_positive_count,rank300_negative_count,rank300_sum,rank300_abs_sum,rank300_log1p_abs_sum,rank300_mean,rank300_std,rank300_min,rank300_max,rank300_q25,rank300_q50,rank300_q75,rank300_l2_norm
0,24,0.04,22,2,7771.362793,7775.362793,8.958844,310.854523,1507.327637,-1.000000,7695.075195,0.041662,0.447167,0.635555,7695.237793,47,0.06,44,3,22343.705078,22349.705078,10.014613,446.874115,2121.170410,-1.0,13230.237305,0.071204,0.473780,0.901704,15328.177734,93,0.07,87,6,28961.251953,28973.251953,10.274163,289.612518,1590.642822,-1.0,13230.237305,0.046763,0.490008,5.156798,16167.934570,267,0.11,256,11,63396.367188,63418.367188,11.057525,211.321228,1151.004028,-1.0,13230.237305,0.191963,1.286384,12.056539,20269.167969
1,25,0.00,22,3,7770.909180,7776.909180,8.959043,310.836365,1507.331299,-1.000000,7695.075195,0.169424,0.460173,0.635555,7695.237793,49,0.02,44,5,22342.330078,22352.330078,10.014730,446.846588,2121.176270,-1.0,13230.237305,0.182074,0.485254,0.972724,15328.177734,96,0.04,87,9,28958.740234,28976.740234,10.274283,289.587402,1590.647949,-1.0,13230.237305,0.027204,0.509039,5.156798,16167.934570,279,0.07,254,25,63378.984375,63428.984375,11.057692,211.263275,1151.013916,-1.0,13230.237305,0.153389,1.193359,12.056539,20269.167969
2,25,0.00,25,0,37840.718750,37840.718750,10.541167,1513.628784,7360.479492,0.002973,37572.000000,0.071942,0.336947,0.930185,37572.503906,49,0.02,49,0,101704.515625,101704.515625,11.529837,2034.090332,9062.647461,0.0,53508.000000,0.082437,0.543660,10.250000,65676.898438,87,0.13,87,0,154790.468750,154790.468750,11.949834,1547.904663,7512.475586,0.0,53508.000000,0.026211,0.452458,9.000000,76702.851562,198,0.34,197,1,291891.000000,291893.000000,12.584146,972.969971,5815.401855,-1.0,53508.000000,0.000000,0.452458,15.000000,102125.882812
3,24,0.04,24,0,5113.334473,5113.334473,8.539803,204.533386,960.186707,0.000000,4905.000000,0.072165,0.270000,0.968463,4908.645996,44,0.12,44,0,19779.242188,19779.242188,9.892439,395.584839,1506.490234,0.0,8144.000000,0.035192,0.324772,1.000000,11013.627930,74,0.26,72,2,19947.607422,19951.607422,9.901115,199.476074,1083.182861,-1.0,8144.000000,0.000000,0.099296,0.868855,11013.969727,129,0.57,124,5,44244.097656,44254.097656,10.697726,147.480331,945.233582,-1.0,8144.000000,0.000000,0.000000,0.440299,16570.025391
4,25,0.00,25,0,60362.050781,60362.050781,11.008132,2414.481934,11785.882812,0.000483,60153.000000,0.269231,0.433735,1.000000,60153.292969,46,0.08,46,0,136685.078125,136685.078125,11.825442,2733.701660,12807.410156,0.0,70187.000000,0.159629,0.514944,1.203327,92602.078125,77,0.23,77,0,139512.812500,139512.812500,11.845919,1395.128174,9155.729492,0.0,70187.000000,0.002044,0.299617,1.179032,92614.085938,165,0.45,165,0,360277.937500,360277.937500,12.794634,1200.926514,8667.953125,0.0,70242.000000,0.000000,0.023322,1.404145,151567.593750


## 14. Peak flags: поиск «магических» значений

Иногда target связан не с тем, что признак большой/маленький, а с тем, что он попал в конкретное повторяющееся значение. Это особенно актуально для счётчиков, категорий, bucketized-признаков и анонимных behavioral-фичей.

Пример логики:

- берём признак;
- группируем строки по его значению;
- считаем target rate в каждом значении;
- ищем значения с достаточной поддержкой и большим lift относительно базового target rate.

Чтобы не переобучиться, используем фильтры по минимальному числу positive и total rows. Эти флаги потом лучше проверять через CV.

In [18]:
def discover_peak_values(
    df: pd.DataFrame,
    y_col: str,
    candidate_features: List[str],
    top_features: int = 250,
    min_pos_count: int = 6,
    min_total_count: int = 10,
    min_lift: float = 5.0,
    max_nunique: int = 3000,
    max_values_per_feature: int = 3,
) -> pd.DataFrame:
    base_rate = df[y_col].mean()
    rows = []
    candidates = [c for c in candidate_features[:top_features] if c in df.columns]

    for i, col in enumerate(candidates):
        nunique = df[col].nunique(dropna=False)
        if nunique > max_nunique:
            continue
        g = df.groupby(col, dropna=False)[y_col].agg(["count", "sum", "mean"]).reset_index()
        g = g[(g["count"] >= min_total_count) & (g["sum"] >= min_pos_count)].copy()
        if g.empty:
            continue
        g["lift"] = g["mean"] / max(base_rate, 1e-12)
        g = g[g["lift"] >= min_lift]
        if g.empty:
            continue
        g["score"] = g["sum"] * np.log1p(g["lift"]) * np.log1p(g["count"])
        g = g.sort_values("score", ascending=False).head(max_values_per_feature)
        for _, r in g.iterrows():
            rows.append({
                "feature": col,
                "value": r[col],
                "count": int(r["count"]),
                "pos_count": int(r["sum"]),
                "target_rate": float(r["mean"]),
                "lift": float(r["lift"]),
                "score": float(r["score"]),
                "nunique": int(nunique),
            })

    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True) if rows else pd.DataFrame()


def make_peak_flags(df: pd.DataFrame, peak_df: pd.DataFrame, atol: float = 1e-6) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    if peak_df is None or len(peak_df) == 0:
        return out
    for i, r in peak_df.iterrows():
        col = r["feature"]
        val = r["value"]
        if col not in df.columns:
            continue
        name = f"peak_{col}_{i:03d}"
        # np.isclose нужен для float32/float64 нюансов.
        out[name] = np.isclose(df[col].values, val, rtol=0, atol=atol).astype("int8")
    if out.shape[1] > 0:
        out["peak_flags_sum"] = out.sum(axis=1).astype("int16")
    return out

if RUN_PEAK_FLAGS:
    peak_values = discover_peak_values(
        train,
        TARGET_COL,
        ranked_features_pruned,
        top_features=PEAK_TOP_FEATURES,
        min_pos_count=PEAK_MIN_POS_COUNT,
        min_total_count=PEAK_MIN_TOTAL_COUNT,
        min_lift=PEAK_MIN_LIFT,
    )
    peak_values.to_csv(ARTIFACT_DIR / "discovered_peak_values.csv", index=False)
    print("Discovered peaks:", len(peak_values))
    display(peak_values.head(50))

    train_peak_flags = make_peak_flags(train, peak_values)
    test_peak_flags = make_peak_flags(test, peak_values)
else:
    peak_values = pd.DataFrame()
    train_peak_flags = pd.DataFrame(index=train.index)
    test_peak_flags = pd.DataFrame(index=test.index)

print(train_peak_flags.shape, test_peak_flags.shape)

Discovered peaks: 4


,feature,value,count,pos_count,target_rate,lift,score,nunique
0,feature_1118,5512.000,114,9,0.078947,5.850788,82.178776,1434
1,feature_1122,0.438,64,6,0.093750,6.947811,51.918437,2788
2,feature_1019,72.000,81,6,0.074074,5.489628,49.448817,2455
3,feature_216,164.000,86,6,0.069767,5.170464,48.761661,2700


(247972, 5) (106274, 5)


## 15. Frequency encoding

Если значение признака часто повторяется, сама частота значения может быть сигналом. Например, популярный bucket может соответствовать массовому поведению, а редкое значение — аномальному.

Это unsupervised-преобразование: target не используется. Поэтому на соревнованиях часто считают частоты по `train + test`. В production обычно считают только по train, чтобы не подглядывать в будущие данные.

In [19]:
def choose_frequency_candidates(profile: pd.DataFrame, ranked_cols: List[str], max_nunique: int, top_n: int) -> List[str]:
    eligible = set(profile.index[profile["nunique_train"] <= max_nunique].tolist())
    return [c for c in ranked_cols if c in eligible][:top_n]


def make_frequency_encoding(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: List[str],
    transductive: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_out = pd.DataFrame(index=train_df.index)
    test_out = pd.DataFrame(index=test_df.index)

    for col in cols:
        if transductive:
            both = pd.concat([train_df[col], test_df[col]], axis=0, ignore_index=True)
            freq = both.value_counts(dropna=False, normalize=True)
        else:
            freq = train_df[col].value_counts(dropna=False, normalize=True)

        train_out[f"freq_{col}"] = train_df[col].map(freq).fillna(0).astype("float32")
        test_out[f"freq_{col}"] = test_df[col].map(freq).fillna(0).astype("float32")

    return train_out, test_out

if RUN_FREQUENCY_ENCODING:
    freq_candidates = choose_frequency_candidates(feature_profile, ranked_features_pruned, FREQ_MAX_NUNIQUE, FREQ_TOP_N)
    print("Frequency encoding candidates:", len(freq_candidates))
    print(freq_candidates[:30])
    train_freq, test_freq = make_frequency_encoding(train, test, freq_candidates, transductive=TRANSDUCTIVE_FREQ_ENCODING)
else:
    freq_candidates = []
    train_freq = pd.DataFrame(index=train.index)
    test_freq = pd.DataFrame(index=test.index)

print(train_freq.shape, test_freq.shape)
display(train_freq.head())

Frequency encoding candidates: 150
['feature_1110', 'feature_1202', 'feature_1115', 'feature_481', 'feature_1205', 'feature_4', 'feature_346', 'feature_1118', 'feature_6', 'feature_333', 'feature_85', 'feature_303', 'feature_62', 'feature_482', 'feature_1230', 'feature_9', 'feature_1116', 'feature_371', 'feature_1112', 'feature_117', 'feature_1193', 'feature_213', 'feature_318', 'feature_1120', 'feature_221', 'feature_341', 'feature_82', 'feature_379', 'feature_277', 'feature_337']
(247972, 150) (106274, 150)


,freq_feature_1110,freq_feature_1202,freq_feature_1115,freq_feature_481,freq_feature_1205,freq_feature_4,freq_feature_346,freq_feature_1118,freq_feature_6,freq_feature_333,freq_feature_85,freq_feature_303,freq_feature_62,freq_feature_482,freq_feature_1230,freq_feature_9,freq_feature_1116,freq_feature_371,freq_feature_1112,freq_feature_117,freq_feature_1193,freq_feature_213,freq_feature_318,freq_feature_1120,freq_feature_221,freq_feature_341,freq_feature_82,freq_feature_379,freq_feature_277,freq_feature_337,freq_feature_86,freq_feature_196,freq_feature_290,freq_feature_286,freq_feature_1114,freq_feature_8,freq_feature_184,freq_feature_7,freq_feature_1195,freq_feature_1200,freq_feature_464,freq_feature_1117,freq_feature_294,freq_feature_56,freq_feature_331,freq_feature_168,freq_feature_1119,freq_feature_366,freq_feature_280,freq_feature_343,freq_feature_254,freq_feature_111,freq_feature_148,freq_feature_5,freq_feature_1217,freq_feature_437,freq_feature_257,freq_feature_1113,freq_feature_1194,freq_feature_231,...,freq_feature_134,freq_feature_264,freq_feature_218,freq_feature_438,freq_feature_285,freq_feature_110,freq_feature_262,freq_feature_135,freq_feature_468,freq_feature_278,freq_feature_1309,freq_feature_412,freq_feature_439,freq_feature_330,freq_feature_100,freq_feature_104,freq_feature_116,freq_feature_405,freq_feature_93,freq_feature_180,freq_feature_336,freq_feature_390,freq_feature_217,freq_feature_70,freq_feature_328,freq_feature_84,freq_feature_80,freq_feature_95,freq_feature_97,freq_feature_1199,freq_feature_339,freq_feature_270,freq_feature_1277,freq_feature_105,freq_feature_289,freq_feature_65,freq_feature_96,freq_feature_347,freq_feature_122,freq_feature_1201,freq_feature_1257,freq_feature_1135,freq_feature_288,freq_feature_400,freq_feature_1271,freq_feature_178,freq_feature_1213,freq_feature_317,freq_feature_266,freq_feature_433,freq_feature_393,freq_feature_430,freq_feature_1273,freq_feature_311,freq_feature_60,freq_feature_1209,freq_feature_1220,freq_feature_310,freq_feature_309,freq_feature_1161
0,0.042360,0.423516,0.182949,0.136061,0.609133,0.349130,0.136061,0.124781,0.131747,0.136061,0.136061,0.136061,0.136061,0.136061,0.728025,0.069508,0.771927,0.136061,0.272085,0.136061,0.322920,0.136061,0.136061,0.271464,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.124354,0.127301,0.136061,0.299521,0.025502,0.456036,0.136061,0.272085,0.136061,0.136061,0.136061,0.136061,0.415398,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.202170,0.690221,0.136061,0.136061,0.272057,0.015238,0.136061,...,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.828580,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.341997,0.136061,0.136061,0.814428,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.413365,0.798651,0.597170,0.136061,0.136061,0.812272,0.136061,0.632408,0.136061,0.136061,0.136061,0.136061,0.136061,0.795230,0.136061,0.136061,0.668445,0.699855,0.136061,0.136061,0.563287
1,0.042360,0.149069,0.182949,0.136061,0.149069,0.349130,0.136061,0.124781,0.131747,0.136061,0.136061,0.136061,0.136061,0.136061,0.149069,0.069508,0.771927,0.136061,0.272085,0.136061,0.322920,0.136061,0.136061,0.271464,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.124354,0.705925,0.136061,0.306202,0.149069,0.149069,0.136061,0.272085,0.136061,0.136061,0.136061,0.136061,0.415398,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.202170,0.149069,0.136061,0.136061,0.272057,0.724547,0.136061,...,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.149069,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.136061,0.149069,0.136061,0.136061,0.149069,0.136061,0.136061,0.13

## 16. OOF target encoding

Target encoding — мощная штука, но с высоким риском leakage. Нельзя просто взять средний target по значению признака на всём train и подставить обратно в train: модель увидит ответ.

Правильная схема:

- для каждой валидационной части считаем mapping только на train-части фолда;
- применяем его к validation-части;
- для test mapping считаем на всём train;
- используем smoothing, чтобы редкие значения не давали экстремальные оценки.

Target encoding имеет смысл только для повторяющихся/дискретных признаков. Для почти непрерывных колонок он будет шумным.

In [20]:
def make_oof_target_encoding(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    y: np.ndarray,
    cols: List[str],
    cv_splits: List[Tuple[np.ndarray, np.ndarray]],
    smoothing: float = 30.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    global_mean = float(np.mean(y))
    train_out = pd.DataFrame(index=train_df.index)
    test_out = pd.DataFrame(index=test_df.index)

    for col in cols:
        encoded = np.full(len(train_df), global_mean, dtype=np.float32)
        for tr_idx, va_idx in cv_splits:
            tmp = pd.DataFrame({col: train_df.iloc[tr_idx][col].values, "target": y[tr_idx]})
            stats = tmp.groupby(col)["target"].agg(["sum", "count"])
            smooth = (stats["sum"] + global_mean * smoothing) / (stats["count"] + smoothing)
            encoded[va_idx] = train_df.iloc[va_idx][col].map(smooth).fillna(global_mean).astype("float32").values

        # Mapping для test по всему train.
        tmp_full = pd.DataFrame({col: train_df[col].values, "target": y})
        stats_full = tmp_full.groupby(col)["target"].agg(["sum", "count"])
        smooth_full = (stats_full["sum"] + global_mean * smoothing) / (stats_full["count"] + smoothing)

        train_out[f"te_{col}"] = encoded
        test_out[f"te_{col}"] = test_df[col].map(smooth_full).fillna(global_mean).astype("float32")

    return train_out, test_out

if RUN_TARGET_ENCODING:
    # Берём пересечение топовых и достаточно дискретных признаков.
    te_candidates = choose_frequency_candidates(feature_profile, ranked_features_pruned, max_nunique=1000, top_n=TARGET_ENCODING_TOP_N)
    print("Target encoding candidates:", len(te_candidates))
    print(te_candidates[:30])

    train_te, test_te = make_oof_target_encoding(
        train, test, y, te_candidates, cv_splits, smoothing=30.0
    )
else:
    te_candidates = []
    train_te = pd.DataFrame(index=train.index)
    test_te = pd.DataFrame(index=test.index)

print(train_te.shape, test_te.shape)
display(train_te.head())

Target encoding candidates: 80
['feature_1110', 'feature_1202', 'feature_1115', 'feature_1205', 'feature_4', 'feature_6', 'feature_85', 'feature_303', 'feature_1230', 'feature_9', 'feature_1116', 'feature_1112', 'feature_1193', 'feature_318', 'feature_1120', 'feature_221', 'feature_341', 'feature_82', 'feature_277', 'feature_337', 'feature_290', 'feature_1114', 'feature_8', 'feature_184', 'feature_7', 'feature_1195', 'feature_1200', 'feature_464', 'feature_1117', 'feature_1119']
(247972, 80) (106274, 80)


,te_feature_1110,te_feature_1202,te_feature_1115,te_feature_1205,te_feature_4,te_feature_6,te_feature_85,te_feature_303,te_feature_1230,te_feature_9,te_feature_1116,te_feature_1112,te_feature_1193,te_feature_318,te_feature_1120,te_feature_221,te_feature_341,te_feature_82,te_feature_277,te_feature_337,te_feature_290,te_feature_1114,te_feature_8,te_feature_184,te_feature_7,te_feature_1195,te_feature_1200,te_feature_464,te_feature_1117,te_feature_1119,te_feature_343,te_feature_148,te_feature_5,te_feature_1217,te_feature_437,te_feature_257,te_feature_1113,te_feature_1194,te_feature_231,te_feature_1244,te_feature_367,te_feature_78,te_feature_151,te_feature_236,te_feature_1245,te_feature_1208,te_feature_1111,te_feature_314,te_feature_124,te_feature_103,te_feature_248,te_feature_155,te_feature_153,te_feature_287,te_feature_296,te_feature_1223,te_feature_134,te_feature_264,te_feature_438,te_feature_262,te_feature_468,te_feature_1309,te_feature_412,te_feature_439,te_feature_330,te_feature_100,te_feature_104,te_feature_405,te_feature_336,te_feature_217,te_feature_84,te_feature_80,te_feature_95,te_feature_97,te_feature_1199,te_feature_270,te_feature_1277,te_feature_105,te_feature_289,te_feature_65
0,0.011044,0.010510,0.015235,0.011856,0.011884,0.014201,0.014759,0.014759,0.012461,0.016012,0.013025,0.014425,0.012131,0.014759,0.014578,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014543,0.011917,0.014759,0.012054,0.012092,0.011589,0.014759,0.014425,0.014936,0.014759,0.014759,0.016403,0.012306,0.014759,0.014759,0.014455,0.013205,0.014759,0.012592,0.014759,0.014759,0.014759,0.014759,0.012538,0.011880,0.014134,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.012594,0.014759,0.014759,0.014759,0.014759,0.014759,0.012959,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.014759,0.010885,0.014759,0.013417,0.014759,0.014759,0.014759
1,0.011041,0.013595,0.015046,0.013595,0.011715,0.013642,0.013994,0.013994,0.013595,0.015195,0.013118,0.014293,0.011789,0.013994,0.014396,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.014303,0.014336,0.013994,0.013575,0.013595,0.013595,0.013994,0.014293,0.014695,0.013994,0.013994,0.015862,0.013595,0.013994,0.013994,0.014236,0.014304,0.013994,0.013595,0.013994,0.013994,0.013994,0.013994,0.013595,0.013595,0.013851,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013595,0.013994,0.013994,0.013994,0.013994,0.013994,0.013595,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013994,0.013595,0.013994,0.013595,0.013994,0.013994,0.013994
2,0.011884,0.015983,0.012367,0.012235,0.013710,0.012257,0.014634,0.009047,0.012627,0.013604,0.012405,0.015734,0.018908,0.013868,0.014159,0.014239,0.013048,0.013913,0.014187,0.021217,0.014347,0.015808,0.014336,0.014643,0.014674,0.017194,0.015327,0.013902,0.011697,0.009845,0.007754,0.013752,0.013049,0.012575,0.013814,0.010680,0.012051,0.014304,0.008288,0.020509,0.006574,0.013888,0.014237,0.014162,0.017637,0.015948,0.013851,0.013940,0.010080,0.013752,0.013690,0.013553,0.013612,0.014093,0.013860,0.012770,0.013771,0.013736,0.013972,0.008996,0.014227,0.013163,0.013701,0.013838,0.014032,0.013686,0.014190,0.013384,0.011906,0.014054,0.014139,0.013819,0.013991,0.013760,0.014913,0.013093,0.013598,0.013787,0.013701,0.013948
3,0.011884,0.010931,0.012367,0.012235,0.013710,0.012257,0.011417,0.014635,0.012627,0.016862,0.013118,0.014293,0.011789,0.013868,0.014159,0.014239,0.013048,0.013913,0.014187,0.012672,0.014347,0.024291,0.013280,0.014643,0.011722,0.013110,0.011859,0.013902,0.014293,0.014695,0.013927,0.013752,0.013049,0.012575,0.013814,0.014401,0.014236,0.012078,0.014429,0.012780,0.013722,0.013888,0.014237,0.014162,0.012958,0.012240,0.012573,0.013940,0.011648,0.013752,0.013690,0.013553,0.013612,0.014093,0.013860,0.012770,0.013771,0.013736,0.013972,0.012625,0.014362,0.013163,0.013701,0.013838,0.014032,0.013686,0.014190,0.013384,0.012859,0.014

## 17. Pairwise interactions

Бустинг умеет строить взаимодействия сам, но иногда явные простые комбинации помогают, особенно если важный сигнал находится в отношениях между фичами.

Добавляем только для небольшого числа топовых признаков, иначе получим комбинаторный взрыв.

Генерируем:

- сумма двух признаков;
- разность `a - b`;
- лог-разность масштабов `log1p(abs(a)) - log1p(abs(b))`;
- флаг, что оба признака ненулевые.

In [21]:
def make_pairwise_interactions(
    df: pd.DataFrame,
    cols: List[str],
    prefix: str = "inter",
) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    cols = [c for c in cols if c in df.columns]
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a_col, b_col = cols[i], cols[j]
            a = df[a_col].astype("float32").values
            b = df[b_col].astype("float32").values
            short_a = a_col.replace("feature_", "f")
            short_b = b_col.replace("feature_", "f")
            base = f"{prefix}_{short_a}_{short_b}"

            out[f"{base}_sum"] = (a + b).astype("float32")
            out[f"{base}_diff"] = (a - b).astype("float32")
            out[f"{base}_logabsdiff"] = (np.log1p(np.abs(a)) - np.log1p(np.abs(b))).astype("float32")
            out[f"{base}_both_nonzero"] = ((a != 0) & (b != 0)).astype("int8")
    return out

if RUN_INTERACTIONS:
    interaction_cols = ranked_features_pruned[:INTERACTION_TOP_N]
    print("Interaction base columns:", interaction_cols)
    train_inter = make_pairwise_interactions(train, interaction_cols, prefix="inter")
    test_inter = make_pairwise_interactions(test, interaction_cols, prefix="inter")
else:
    interaction_cols = []
    train_inter = pd.DataFrame(index=train.index)
    test_inter = pd.DataFrame(index=test.index)

print(train_inter.shape, test_inter.shape)
display(train_inter.head())

Interaction base columns: ['feature_1110', 'feature_16', 'feature_34', 'feature_41', 'feature_1094', 'feature_1202', 'feature_43', 'feature_1087', 'feature_1122', 'feature_29', 'feature_1079', 'feature_13', 'feature_1034', 'feature_1107']
(247972, 364) (106274, 364)


,inter_f1110_f16_sum,inter_f1110_f16_diff,inter_f1110_f16_logabsdiff,inter_f1110_f16_both_nonzero,inter_f1110_f34_sum,inter_f1110_f34_diff,inter_f1110_f34_logabsdiff,inter_f1110_f34_both_nonzero,inter_f1110_f41_sum,inter_f1110_f41_diff,inter_f1110_f41_logabsdiff,inter_f1110_f41_both_nonzero,inter_f1110_f1094_sum,inter_f1110_f1094_diff,inter_f1110_f1094_logabsdiff,inter_f1110_f1094_both_nonzero,inter_f1110_f1202_sum,inter_f1110_f1202_diff,inter_f1110_f1202_logabsdiff,inter_f1110_f1202_both_nonzero,inter_f1110_f43_sum,inter_f1110_f43_diff,inter_f1110_f43_logabsdiff,inter_f1110_f43_both_nonzero,inter_f1110_f1087_sum,inter_f1110_f1087_diff,inter_f1110_f1087_logabsdiff,inter_f1110_f1087_both_nonzero,inter_f1110_f1122_sum,inter_f1110_f1122_diff,inter_f1110_f1122_logabsdiff,inter_f1110_f1122_both_nonzero,inter_f1110_f29_sum,inter_f1110_f29_diff,inter_f1110_f29_logabsdiff,inter_f1110_f29_both_nonzero,inter_f1110_f1079_sum,inter_f1110_f1079_diff,inter_f1110_f1079_logabsdiff,inter_f1110_f1079_both_nonzero,inter_f1110_f13_sum,inter_f1110_f13_diff,inter_f1110_f13_logabsdiff,inter_f1110_f13_both_nonzero,inter_f1110_f1034_sum,inter_f1110_f1034_diff,inter_f1110_f1034_logabsdiff,inter_f1110_f1034_both_nonzero,inter_f1110_f1107_sum,inter_f1110_f1107_diff,inter_f1110_f1107_logabsdiff,inter_f1110_f1107_both_nonzero,inter_f16_f34_sum,inter_f16_f34_diff,inter_f16_f34_logabsdiff,inter_f16_f34_both_nonzero,inter_f16_f41_sum,inter_f16_f41_diff,inter_f16_f41_logabsdiff,inter_f16_f41_both_nonzero,...,inter_f1122_f29_sum,inter_f1122_f29_diff,inter_f1122_f29_logabsdiff,inter_f1122_f29_both_nonzero,inter_f1122_f1079_sum,inter_f1122_f1079_diff,inter_f1122_f1079_logabsdiff,inter_f1122_f1079_both_nonzero,inter_f1122_f13_sum,inter_f1122_f13_diff,inter_f1122_f13_logabsdiff,inter_f1122_f13_both_nonzero,inter_f1122_f1034_sum,inter_f1122_f1034_diff,inter_f1122_f1034_logabsdiff,inter_f1122_f1034_both_nonzero,inter_f1122_f1107_sum,inter_f1122_f1107_diff,inter_f1122_f1107_logabsdiff,inter_f1122_f1107_both_nonzero,inter_f29_f1079_sum,inter_f29_f1079_diff,inter_f29_f1079_logabsdiff,inter_f29_f1079_both_nonzero,inter_f29_f13_sum,inter_f29_f13_diff,inter_f29_f13_logabsdiff,inter_f29_f13_both_nonzero,inter_f29_f1034_sum,inter_f29_f1034_diff,inter_f29_f1034_logabsdiff,inter_f29_f1034_both_nonzero,inter_f29_f1107_sum,inter_f29_f1107_diff,inter_f29_f1107_logabsdiff,inter_f29_f1107_both_nonzero,inter_f1079_f13_sum,inter_f1079_f13_diff,inter_f1079_f13_logabsdiff,inter_f1079_f13_both_nonzero,inter_f1079_f1034_sum,inter_f1079_f1034_diff,inter_f1079_f1034_logabsdiff,inter_f1079_f1034_both_nonzero,inter_f1079_f1107_sum,inter_f1079_f1107_diff,inter_f1079_f1107_logabsdiff,inter_f1079_f1107_both_nonzero,inter_f13_f1034_sum,inter_f13_f1034_diff,inter_f13_f1034_logabsdiff,inter_f13_f1034_both_nonzero,inter_f13_f1107_sum,inter_f13_f1107_diff,inter_f13_f1107_logabsdiff,inter_f13_f1107_both_nonzero,inter_f1034_f1107_sum,inter_f1034_f1107_diff,inter_f1034_f1107_logabsdiff,inter_f1034_f1107_both_nonzero
0,-0.451624,-1.548376,0.255940,1,-0.479362,-1.520638,0.274017,1,-0.264298,-1.735702,0.141735,1,-0.818167,-1.181833,0.526081,1,-1.0,-1.0,0.693147,0,-0.604903,-1.395097,0.360184,1,-0.779976,-1.220024,0.494277,1,-0.662982,-1.337018,0.402705,1,-0.109024,-1.890976,0.056054,1,-0.993855,-1.006145,0.687021,1,-0.472698,-1.527302,0.269645,1,7694.075195,-7696.075195,-8.255318,1,-0.958338,-1.041662,0.652329,1,1.069014,0.027738,0.018077,1,1.284078,-0.187326,-0.114205,1,...,1.227994,-0.553957,-0.346651,1,0.343164,0.330873,0.284315,1,0.864320,-0.190283,-0.133061,1,7695.412109,-7694.738281,-8.658024,1,0.378680,0.295356,0.249624,1,0.897121,0.884830,0.630966,1,1.418277,0.363674,0.213590,1,7695.966309,-7694.184082,-8.311373,1,0.932638,0.849313,0.596275,1,0.533447,-0.521156,-0.417376,1,7695.081543,-7695.068848,-8.942339,1,0.047808,-0.035517,-0.034691,1,7695.602539,-7694.547852,-8.524962,1,0.568964,0.485639,0.382685,1,7695.116699,7695.033691,8.907648,1
1,-0.451624,-1.548376,0.255940,1,-0.479362,-1.520638,0.2740

## 18. PCA и SVD

Пробуем два разных способа получить компоненты:

1. **PCA по значениям топовых признаков**. Перед PCA делаем `log1p(abs(x)) * sign(x)`, чтобы огромные значения не доминировали.
2. **TruncatedSVD по nonzero-mask**. Это отдельный взгляд: не «какие значения», а «какие признаки вообще активны».

Ожидания:

- для LightGBM/CatBoost PCA может дать небольшой плюс или ноль;
- для Logistic Regression / Neural Network / stacking PCA/SVD часто полезнее;
- SVD по маске нулей может быть очень полезным, если структура активности важна.

In [22]:
def signed_log1p(df: pd.DataFrame) -> np.ndarray:
    arr = df.replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32, copy=False)
    return np.sign(arr) * np.log1p(np.abs(arr))


def make_pca_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: List[str],
    n_components: int = 32,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, PCA]:
    X_train = signed_log1p(train_df[cols])
    X_test = signed_log1p(test_df[cols])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=n_components, random_state=random_state)
    train_comp = pca.fit_transform(X_train_scaled)
    test_comp = pca.transform(X_test_scaled)

    names = [f"pca_value_{i:02d}" for i in range(train_comp.shape[1])]
    train_out = pd.DataFrame(train_comp.astype("float32"), columns=names, index=train_df.index)
    test_out = pd.DataFrame(test_comp.astype("float32"), columns=names, index=test_df.index)

    return train_out, test_out, pca


def make_nonzero_svd_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: List[str],
    n_components: int = 32,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, TruncatedSVD]:
    if not HAS_SCIPY:
        raise ImportError("scipy is not installed")
    X_train_mask = sp.csr_matrix((train_df[cols].values != 0).astype(np.float32))
    X_test_mask = sp.csr_matrix((test_df[cols].values != 0).astype(np.float32))

    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    train_comp = svd.fit_transform(X_train_mask)
    test_comp = svd.transform(X_test_mask)

    names = [f"svd_nonzero_{i:02d}" for i in range(train_comp.shape[1])]
    train_out = pd.DataFrame(train_comp.astype("float32"), columns=names, index=train_df.index)
    test_out = pd.DataFrame(test_comp.astype("float32"), columns=names, index=test_df.index)

    return train_out, test_out, svd

if RUN_PCA_SVD:
    # Для PCA берём топ-500, чтобы снизить шум и ускорить расчёт.
    pca_cols = [c for c in ranked_features_pruned[:500] if c in train.columns]
    n_pca = min(PCA_N_COMPONENTS, len(pca_cols), len(train) - 1)
    print("PCA columns:", len(pca_cols), "components:", n_pca)
    train_pca, test_pca, pca_model = make_pca_features(train, test, pca_cols, n_components=n_pca, random_state=RANDOM_STATE)

    print("PCA explained variance ratio sum:", float(pca_model.explained_variance_ratio_.sum()))

    # Для SVD по маске можно брать все good_base_cols: матрица sparse.
    if HAS_SCIPY:
        n_svd = min(SVD_N_COMPONENTS, len(good_base_cols) - 1)
        train_svd, test_svd, svd_model = make_nonzero_svd_features(train, test, good_base_cols, n_components=n_svd, random_state=RANDOM_STATE)
        print("SVD explained variance ratio sum:", float(svd_model.explained_variance_ratio_.sum()))
    else:
        print("scipy not installed, skipping nonzero SVD")
        train_svd = pd.DataFrame(index=train.index)
        test_svd = pd.DataFrame(index=test.index)
else:
    train_pca = pd.DataFrame(index=train.index)
    test_pca = pd.DataFrame(index=test.index)
    train_svd = pd.DataFrame(index=train.index)
    test_svd = pd.DataFrame(index=test.index)

print(train_pca.shape, train_svd.shape)
display(pd.concat([train_pca.head(), train_svd.head()], axis=1).head())

PCA columns: 500 components: 32
PCA explained variance ratio sum: 0.5713316202163696
SVD explained variance ratio sum: 0.7925409078598022
(247972, 32) (247972, 32)


,pca_value_00,pca_value_01,pca_value_02,pca_value_03,pca_value_04,pca_value_05,pca_value_06,pca_value_07,pca_value_08,pca_value_09,pca_value_10,pca_value_11,pca_value_12,pca_value_13,pca_value_14,pca_value_15,pca_value_16,pca_value_17,pca_value_18,pca_value_19,pca_value_20,pca_value_21,pca_value_22,pca_value_23,pca_value_24,pca_value_25,pca_value_26,pca_value_27,pca_value_28,pca_value_29,pca_value_30,pca_value_31,svd_nonzero_00,svd_nonzero_01,svd_nonzero_02,svd_nonzero_03,svd_nonzero_04,svd_nonzero_05,svd_nonzero_06,svd_nonzero_07,svd_nonzero_08,svd_nonzero_09,svd_nonzero_10,svd_nonzero_11,svd_nonzero_12,svd_nonzero_13,svd_nonzero_14,svd_nonzero_15,svd_nonzero_16,svd_nonzero_17,svd_nonzero_18,svd_nonzero_19,svd_nonzero_20,svd_nonzero_21,svd_nonzero_22,svd_nonzero_23,svd_nonzero_24,svd_nonzero_25,svd_nonzero_26,svd_nonzero_27,svd_nonzero_28,svd_nonzero_29,svd_nonzero_30,svd_nonzero_31
0,10.716595,-6.015415,-12.717984,-0.183566,-1.959709,-0.388098,0.732493,-1.509707,-0.989250,-1.657119,-0.598362,-0.145977,1.182859,-1.147979,0.784941,0.717769,-0.634389,-1.130526,0.073454,0.493595,-0.245710,0.363811,-0.372266,-0.130842,0.850535,0.123548,0.875994,0.599024,0.349553,0.220738,1.039048,-0.620564,26.275352,-19.277481,-6.190525,3.411335,-0.943245,-1.684416,0.837881,0.100656,-0.229628,-0.059236,0.105199,-0.286916,0.084835,-0.200558,-0.258512,0.114760,0.212376,0.133254,-0.235052,0.077365,0.126259,-0.024608,-0.214171,-0.129120,-0.187390,-0.150126,-0.025661,0.134582,0.051428,-0.007166,-0.175648,-0.013873
1,13.084078,-18.275223,-5.530946,-1.069400,-2.712479,2.827191,-0.618344,-1.990866,-0.744924,-0.596232,-0.153750,0.282555,0.498528,-1.243159,-0.265524,0.120750,-0.237522,-1.579199,0.473289,0.698094,-1.010576,0.584863,-0.043834,1.185028,0.336814,-0.789452,-0.038469,0.853677,1.057246,-0.020056,1.525421,0.018371,29.145529,-19.848976,5.078732,1.523319,1.438481,0.768832,-0.298340,-0.061594,0.031683,-0.004620,-0.013489,0.058566,-0.000053,-0.087836,0.170389,-0.018913,-0.432504,0.088811,0.152300,0.052626,-0.011512,0.133620,0.137596,0.037146,0.068881,0.546096,-0.050009,-0.340863,-0.345803,0.049307,-0.025606,0.130805
2,3.157341,6.507373,1.503831,-1.117761,-3.073986,-1.215343,0.060036,0.971137,-4.910412,-0.560554,6.014725,2.845772,-0.766928,1.466896,-2.064497,0.316693,-3.727611,2.347825,-2.436436,1.553969,0.878945,3.709176,-3.138179,-3.064739,-2.650337,-3.953435,-2.965316,-2.035737,-2.391546,-0.754304,0.787095,1.470177,12.525258,5.957142,-1.486451,-0.332389,0.915476,-0.756954,-1.670918,-0.900253,-0.842943,-2.684641,-0.351030,-1.052139,0.179224,0.308165,1.693024,0.649639,-0.033624,0.316760,-0.538671,-0.310992,-0.478845,-0.407604,0.598846,-0.447276,0.451424,0.857156,-0.184433,0.015237,-1.899933,-0.671538,-0.307045,0.238064
3,-8.513441,-1.331978,-2.881840,1.365599,0.226574,-3.305436,-3.745111,0.261498,1.725444,-1.948184,0.746845,1.566494,-1.169172,0.831898,1.309209,-1.567067,-0.021736,0.415745,1.883724,-1.331111,-0.077918,-1.616425,-0.566245,1.213989,-1.430381,-0.187016,-1.015168,-1.103778,-1.958251,-0.887897,-0.591590,1.207779,8.919265,5.090974,0.018081,2.375796,-3.253805,-1.231613,2.118627,-0.983472,-0.544608,-0.475275,-0.335507,1.659178,0.384070,0.092106,-0.435797,-0.298874,-0.042392,0.151570,-0.244393,-0.761788,0.038249,0.631394,-0.403204,-0.726366,1.245443,1.154077,-0.366909,-0.093469,-0.244746,-0.092746,-0.696260,0.700136
4,-3.512703,5.710971,0.303252,8.821564,-2.172427,5.290529,-4.312706,-2.964363,4.736632,-0.088456,0.599794,-0.604725,2.061606,0.009256,-1.510850,1.130264,-0.625920,2.333865,1.582871,0.063707,-0.084127,-2.999458,-1.004662,2.526076,0.389135,0.329808,0.593812,0.337313,0.087530,2.019962,-0.246097,-0.579891,11.889440,6.495407,-0.578235,1.581379,0.555172,2.488414,3.546805,-0.456115,1.629623,-0.610156,0.420854,1.321077,0.882822,-1.013623,0.135625,0.646899,0.041615,-0.278321,-0.419697,-0.650954,2.117695,1.262456,0.952513,-0.164754,0.582781,-0.976252,-0.231023,0.040614,-0.193714,-1.120048,0.210037,-0.892747


## 19. KMeans-сегменты по PCA-пространству

Иногда полезно дать модели «тип строки» / сегмент поведения. Для этого кластеризуем PCA-компоненты и добавляем:

- номер ближайшего кластера;
- расстояние до ближайшего центра;
- расстояние до нескольких центров.

Это не всегда даёт плюс, но как дополнительный сигнал для ансамбля стоит проверить.

In [23]:
def make_kmeans_features(
    train_comp: pd.DataFrame,
    test_comp: pd.DataFrame,
    n_clusters: int = 16,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, MiniBatchKMeans]:
    if train_comp.shape[1] == 0:
        return pd.DataFrame(index=train_comp.index), pd.DataFrame(index=test_comp.index), None

    km = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=4096,
        n_init="auto",
    )
    km.fit(train_comp.values)

    train_dist = km.transform(train_comp.values).astype("float32")
    test_dist = km.transform(test_comp.values).astype("float32")
    train_cluster = train_dist.argmin(axis=1).astype("int16")
    test_cluster = test_dist.argmin(axis=1).astype("int16")

    train_out = pd.DataFrame(index=train_comp.index)
    test_out = pd.DataFrame(index=test_comp.index)
    train_out["kmeans_cluster"] = train_cluster
    test_out["kmeans_cluster"] = test_cluster
    train_out["kmeans_min_dist"] = train_dist.min(axis=1)
    test_out["kmeans_min_dist"] = test_dist.min(axis=1)

    # Дистанции до центров — числовые признаки. Их немного, поэтому оставляем все.
    for i in range(n_clusters):
        train_out[f"kmeans_dist_{i:02d}"] = train_dist[:, i]
        test_out[f"kmeans_dist_{i:02d}"] = test_dist[:, i]

    return train_out, test_out, km

if RUN_KMEANS_ON_PCA and train_pca.shape[1] > 0:
    train_kmeans, test_kmeans, kmeans_model = make_kmeans_features(
        train_pca, test_pca, n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE
    )
else:
    train_kmeans = pd.DataFrame(index=train.index)
    test_kmeans = pd.DataFrame(index=test.index)

print(train_kmeans.shape, test_kmeans.shape)
display(train_kmeans.head())

(247972, 18) (106274, 18)


,kmeans_cluster,kmeans_min_dist,kmeans_dist_00,kmeans_dist_01,kmeans_dist_02,kmeans_dist_03,kmeans_dist_04,kmeans_dist_05,kmeans_dist_06,kmeans_dist_07,kmeans_dist_08,kmeans_dist_09,kmeans_dist_10,kmeans_dist_11,kmeans_dist_12,kmeans_dist_13,kmeans_dist_14,kmeans_dist_15
0,7,3.328536,23.201359,15.784491,21.673519,20.692768,22.357592,24.873621,23.368662,3.328536,18.482475,20.121256,26.685623,21.790684,20.624222,23.982521,27.710909,21.414524
1,1,4.418838,29.463097,4.418838,28.171837,27.018633,29.080721,30.354660,17.837002,16.982367,25.280920,26.863173,31.671101,28.535477,27.224815,29.956318,23.213877,27.888184
2,4,13.904314,17.787399,30.532282,17.923971,17.283579,13.904314,22.129463,24.973858,23.750450,18.916733,14.014269,26.974209,15.137156,16.078501,18.745480,28.470804,20.115635
3,5,6.836721,13.231807,28.927961,25.565754,9.443208,19.601141,6.836721,23.773344,23.192209,17.939083,16.847971,35.385841,13.524857,20.091972,9.919911,19.539791,24.304493
4,0,13.025826,13.025826,32.345760,21.925795,14.299624,16.523329,16.051733,27.157507,25.504374,22.198687,17.031439,31.866344,15.766615,16.632769,17.611032,26.338995,23.384666


## 20. Сборка финальных feature sets

Мы не пытаемся выбрать один «идеальный» набор. Для ансамбля полезно иметь несколько разных представлений:

- `raw_clean`: исходные хорошие признаки;
- `raw_clean_meta`: исходные + агрегаты строки;
- `top500_engineered`: топ-500 + все engineered-признаки;
- `all_engineered`: все хорошие исходные + engineered;
- `compact_for_linear`: топовые признаки + PCA/SVD/encoding, удобно для логистической регрессии/MLP/stacking.

In [24]:
# Склеиваем engineered-признаки.
engineered_train_parts = [
    train_meta,
    train_block_meta,
    train_topk_meta,
    train_peak_flags,
    train_freq,
    train_te,
    train_inter,
    train_pca,
    train_svd,
    train_kmeans,
]
engineered_test_parts = [
    test_meta,
    test_block_meta,
    test_topk_meta,
    test_peak_flags,
    test_freq,
    test_te,
    test_inter,
    test_pca,
    test_svd,
    test_kmeans,
]

X_train_engineered = pd.concat(engineered_train_parts, axis=1)
X_test_engineered = pd.concat(engineered_test_parts, axis=1)

# Удаляем дублирующиеся имена колонок, если вдруг появились.
X_train_engineered = X_train_engineered.loc[:, ~X_train_engineered.columns.duplicated()].copy()
X_test_engineered = X_test_engineered.loc[:, ~X_test_engineered.columns.duplicated()].copy()

# Синхронизация колонок train/test.
X_test_engineered = X_test_engineered.reindex(columns=X_train_engineered.columns, fill_value=0)

print("Engineered train/test:", X_train_engineered.shape, X_test_engineered.shape)
print("Engineered examples:")
display(X_train_engineered.head())

# Базовые матрицы.
X_train_raw_clean = train[good_base_cols].copy()
X_test_raw_clean = test[good_base_cols].copy()

# Несколько списков колонок.
top300 = [c for c in ranked_features_pruned[:300] if c in X_train_raw_clean.columns]
top500 = [c for c in ranked_features_pruned[:500] if c in X_train_raw_clean.columns]
top800 = [c for c in ranked_features_pruned[:800] if c in X_train_raw_clean.columns]

engineered_cols = X_train_engineered.columns.tolist()
pca_svd_cols = [c for c in engineered_cols if c.startswith("pca_") or c.startswith("svd_") or c.startswith("kmeans_")]
encoding_cols = [c for c in engineered_cols if c.startswith("freq_") or c.startswith("te_")]
meta_cols = [c for c in engineered_cols if not (c.startswith("pca_") or c.startswith("svd_") or c.startswith("freq_") or c.startswith("te_"))]

feature_sets = {
    "raw_clean": good_base_cols,
    "top300_ranked": top300,
    "top500_ranked": top500,
    "top800_ranked": top800,
    "raw_clean_plus_meta": good_base_cols + meta_cols,
    "top500_plus_engineered": top500 + engineered_cols,
    "all_engineered": good_base_cols + engineered_cols,
    "compact_for_linear_or_stack": top500 + encoding_cols + pca_svd_cols + [c for c in meta_cols if "nonzero" in c or "zero_frac" in c or "log1p" in c or "peak" in c],
}

# Удаляем повторы с сохранением порядка.
def unique_keep_order(cols: List[str]) -> List[str]:
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

feature_sets = {name: unique_keep_order(cols) for name, cols in feature_sets.items()}

for name, cols in feature_sets.items():
    print(f"{name:28s}: {len(cols)} columns")

with open(ARTIFACT_DIR / "feature_sets.json", "w") as f:
    json.dump(feature_sets, f, indent=2)
print("Saved:", ARTIFACT_DIR / "feature_sets.json")

Engineered train/test: (247972, 826) (106274, 826)
Engineered examples:


,all_nonzero_count,all_zero_frac,all_positive_count,all_negative_count,all_sum,all_abs_sum,all_log1p_abs_sum,all_mean,all_std,all_min,all_max,all_q25,all_q50,all_q75,all_l2_norm,block_0000_0099_nonzero_count,block_0000_0099_zero_frac,block_0000_0099_log1p_abs_sum,block_0000_0099_max,block_0000_0099_std,block_0100_0199_nonzero_count,block_0100_0199_zero_frac,block_0100_0199_log1p_abs_sum,block_0100_0199_max,block_0100_0199_std,block_0200_0299_nonzero_count,block_0200_0299_zero_frac,block_0200_0299_log1p_abs_sum,block_0200_0299_max,block_0200_0299_std,block_0300_0399_nonzero_count,block_0300_0399_zero_frac,block_0300_0399_log1p_abs_sum,block_0300_0399_max,block_0300_0399_std,block_0400_0499_nonzero_count,block_0400_0499_zero_frac,block_0400_0499_log1p_abs_sum,block_0400_0499_max,block_0400_0499_std,block_0500_0599_nonzero_count,block_0500_0599_zero_frac,block_0500_0599_log1p_abs_sum,block_0500_0599_max,block_0500_0599_std,block_0600_0699_nonzero_count,block_0600_0699_zero_frac,block_0600_0699_log1p_abs_sum,block_0600_0699_max,block_0600_0699_std,block_0700_0799_nonzero_count,block_0700_0799_zero_frac,block_0700_0799_log1p_abs_sum,block_0700_0799_max,block_0700_0799_std,block_0800_0899_nonzero_count,block_0800_0899_zero_frac,block_0800_0899_log1p_abs_sum,block_0800_0899_max,block_0800_0899_std,...,pca_value_22,pca_value_23,pca_value_24,pca_value_25,pca_value_26,pca_value_27,pca_value_28,pca_value_29,pca_value_30,pca_value_31,svd_nonzero_00,svd_nonzero_01,svd_nonzero_02,svd_nonzero_03,svd_nonzero_04,svd_nonzero_05,svd_nonzero_06,svd_nonzero_07,svd_nonzero_08,svd_nonzero_09,svd_nonzero_10,svd_nonzero_11,svd_nonzero_12,svd_nonzero_13,svd_nonzero_14,svd_nonzero_15,svd_nonzero_16,svd_nonzero_17,svd_nonzero_18,svd_nonzero_19,svd_nonzero_20,svd_nonzero_21,svd_nonzero_22,svd_nonzero_23,svd_nonzero_24,svd_nonzero_25,svd_nonzero_26,svd_nonzero_27,svd_nonzero_28,svd_nonzero_29,svd_nonzero_30,svd_nonzero_31,kmeans_cluster,kmeans_min_dist,kmeans_dist_00,kmeans_dist_01,kmeans_dist_02,kmeans_dist_03,kmeans_dist_04,kmeans_dist_05,kmeans_dist_06,kmeans_dist_07,kmeans_dist_08,kmeans_dist_09,kmeans_dist_10,kmeans_dist_11,kmeans_dist_12,kmeans_dist_13,kmeans_dist_14,kmeans_dist_15
0,1124,0.173529,1104,20,95519.625000,95559.625000,11.467516,70.235016,665.595337,-1.0,13230.237305,0.011121,0.252916,1.810390,24682.248047,96,0.04,5.889419,49.332207,8.713109,100,0.00,7.326706,784.520752,92.580345,100,0.00,5.794333,56.324829,9.468415,100,0.00,7.499880,471.928802,52.102371,100,0.00,6.660453,159.094818,22.100138,100,0.00,9.476153,5071.001953,652.406677,100,0.00,6.967532,409.535065,46.792557,100,0.00,9.357634,2498.236328,366.160278,100,0.00,10.873399,13230.237305,2134.547852,...,-0.372266,-0.130842,0.850535,0.123548,0.875994,0.599024,0.349553,0.220738,1.039048,-0.620564,26.275352,-19.277481,-6.190525,3.411335,-0.943245,-1.684416,0.837881,0.100656,-0.229628,-0.059236,0.105199,-0.286916,0.084835,-0.200558,-0.258512,0.114760,0.212376,0.133254,-0.235052,0.077365,0.126259,-0.024608,-0.214171,-0.129120,-0.187390,-0.150126,-0.025661,0.134582,0.051428,-0.007166,-0.175648,-0.013873,7,3.328536,23.201359,15.784491,21.673519,20.692768,22.357592,24.873621,23.368662,3.328536,18.482475,20.121256,26.685623,21.790684,20.624222,23.982521,27.710909,21.414524
1,1282,0.057353,1103,179,95358.242188,95716.242188,11.469153,70.116356,665.609619,-1.0,13230.237305,0.009211,0.255198,1.722785,24682.248047,96,0.04,5.889419,49.332207,8.713109,100,0.00,7.326706,784.520752,92.580345,100,0.00,5.794333,56.324829,9.468415,100,0.00,7.499880,471.928802,52.102371,100,0.00,6.660453,159.094818,22.100138,100,0.00,9.476153,5071.001953,652.406677,100,0.00,6.967532,409.535065,46.792557,100,0.00,9.357634,2498.236328,366.160278,100,0.00,10.873399,13230.237305,2134.547852,...,-0.043834,1.185028,0.336814,-0.789452,-0.038469,0.853677,1.057246,-0.020056,1.525421,0.018371,29.145529,-19.848976,5.078732,1.523319,1.438481,0.768832,-0.298340,-0.061594,0.031683,-0.004620,-0.013489,0.058566,-0.000053,-0.087836,0.170

raw_clean                   : 1360 columns
top300_ranked               : 300 columns
top500_ranked               : 500 columns
top800_ranked               : 800 columns
raw_clean_plus_meta         : 1892 columns
top500_plus_engineered      : 1326 columns
all_engineered              : 2186 columns
compact_for_linear_or_stack : 965 columns
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/feature_sets.json


## 21. Сохранение данных

Сохраняем:

- `X_train_raw_clean` / `X_test_raw_clean`;
- `X_train_engineered` / `X_test_engineered`;
- `y_train`;
- `train_index` / `test_index`;
- таблицы важностей и ранжирования;
- списки feature sets.

Формат: сначала пытаемся сохранить `parquet`; если нет `pyarrow`/`fastparquet`, сохраняем `pickle`.

In [25]:
def save_table(df: pd.DataFrame, path_stem: Path) -> Path:
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = path_stem.with_suffix(".parquet")
    pickle_path = path_stem.with_suffix(".pkl")
    try:
        df.to_parquet(parquet_path, index=False)
        print("Saved parquet:", parquet_path)
        return parquet_path
    except Exception as e:
        print(f"Could not save parquet ({type(e).__name__}: {e}). Saving pickle instead.")
        df.to_pickle(pickle_path)
        print("Saved pickle:", pickle_path)
        return pickle_path

# Важно: ID и target сохраняем отдельно, чтобы случайно не засунуть их в модель как фичи.
pd.Series(y, name=TARGET_COL).to_csv(ARTIFACT_DIR / "y_train.csv", index=False)
train[[ID_COL]].to_csv(ARTIFACT_DIR / "train_index.csv", index=False)
test[[ID_COL]].to_csv(ARTIFACT_DIR / "test_index.csv", index=False)

save_table(X_train_raw_clean, ARTIFACT_DIR / "X_train_raw_clean")
save_table(X_test_raw_clean, ARTIFACT_DIR / "X_test_raw_clean")
save_table(X_train_engineered, ARTIFACT_DIR / "X_train_engineered")
save_table(X_test_engineered, ARTIFACT_DIR / "X_test_engineered")

print("All processed artifacts saved to:", ARTIFACT_DIR)

Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/X_train_raw_clean.parquet
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/X_test_raw_clean.parquet
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/X_train_engineered.parquet
Saved parquet: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data/X_test_engineered.parquet
All processed artifacts saved to: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/modeling/processed_data


## 22. Быстрый benchmark feature sets

Это не финальное обучение. Цель — понять, какие наборы признаков вообще имеют смысл.

Используем один и тот же LightGBM-конфиг и одинаковые CV-фолды. Так сравнение честнее: если `top500_plus_engineered` лучше `top500_ranked`, значит помог именно feature engineering, а не другие гиперпараметры.

In [26]:
def get_matrix_for_feature_set(name: str) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    cols = feature_sets[name]
    raw_cols = [c for c in cols if c in X_train_raw_clean.columns]
    eng_cols = [c for c in cols if c in X_train_engineered.columns]

    Xtr_parts = []
    Xte_parts = []
    if raw_cols:
        Xtr_parts.append(X_train_raw_clean[raw_cols])
        Xte_parts.append(X_test_raw_clean[raw_cols])
    if eng_cols:
        Xtr_parts.append(X_train_engineered[eng_cols])
        Xte_parts.append(X_test_engineered[eng_cols])

    Xtr = pd.concat(Xtr_parts, axis=1)
    Xte = pd.concat(Xte_parts, axis=1)
    Xtr = Xtr.loc[:, ~Xtr.columns.duplicated()].copy()
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    return Xtr, Xte, Xtr.columns.tolist()


def benchmark_lgb_feature_set(
    name: str,
    max_boost_rounds: int = 2500,
    early_stopping_rounds: int = 100,
) -> Dict[str, float]:
    if not HAS_LIGHTGBM:
        print("No LightGBM installed; skipping benchmark.")
        return {}

    Xtr, _, cols = get_matrix_for_feature_set(name)
    scale_pos_weight = (len(y) - y.sum()) / max(y.sum(), 1)

    params = dict(
        objective="binary",
        metric="auc",
        boosting_type="gbdt",
        learning_rate=0.03,
        num_leaves=63,
        min_data_in_leaf=80,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        bagging_freq=1,
        lambda_l1=0.1,
        lambda_l2=10.0,
        scale_pos_weight=scale_pos_weight,
        verbosity=-1,
        seed=RANDOM_STATE,
        feature_pre_filter=False,
        num_threads=max(os.cpu_count() - 1, 1),
    )

    fold_aucs = []
    fold_best_iters = []
    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        print(f"[{name}] fold {fold}")
        dtrain = lgb.Dataset(Xtr.iloc[tr_idx], label=y[tr_idx], free_raw_data=True)
        dvalid = lgb.Dataset(Xtr.iloc[va_idx], label=y[va_idx], reference=dtrain, free_raw_data=True)
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=max_boost_rounds,
            valid_sets=[dvalid],
            valid_names=["valid"],
            callbacks=[
                lgb.early_stopping(early_stopping_rounds, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )
        pred = model.predict(Xtr.iloc[va_idx], num_iteration=model.best_iteration)
        auc = roc_auc_score(y[va_idx], pred)
        fold_aucs.append(auc)
        fold_best_iters.append(model.best_iteration)
        print(f"AUC={auc:.6f}, best_iter={model.best_iteration}")
        del dtrain, dvalid, model
        gc.collect()

    return {
        "feature_set": name,
        "n_cols": len(cols),
        "mean_auc": float(np.mean(fold_aucs)),
        "std_auc": float(np.std(fold_aucs)),
        "mean_best_iteration": float(np.mean(fold_best_iters)),
    }

if RUN_BENCHMARK and HAS_LIGHTGBM:
    benchmark_names = [
        "top300_ranked",
        "top500_ranked",
        "raw_clean",
        "raw_clean_plus_meta",
        "top500_plus_engineered",
        "all_engineered",
        "compact_for_linear_or_stack",
    ]
    benchmark_results = []
    for name in benchmark_names:
        result = benchmark_lgb_feature_set(name)
        if result:
            benchmark_results.append(result)
    benchmark_df = pd.DataFrame(benchmark_results).sort_values("mean_auc", ascending=False)
    benchmark_df.to_csv(ARTIFACT_DIR / "feature_set_benchmark_lgb.csv", index=False)
    display(benchmark_df)
else:
    benchmark_df = pd.DataFrame()
    print("Benchmark skipped.")

[top300_ranked] fold 0
AUC=0.638781, best_iter=187
[top300_ranked] fold 1
AUC=0.645181, best_iter=194
[top300_ranked] fold 2
AUC=0.627652, best_iter=99
[top300_ranked] fold 3
AUC=0.639057, best_iter=120
[top300_ranked] fold 4
AUC=0.640206, best_iter=211
[top500_ranked] fold 0
AUC=0.630081, best_iter=194
[top500_ranked] fold 1
AUC=0.642391, best_iter=164
[top500_ranked] fold 2
AUC=0.624829, best_iter=104
[top500_ranked] fold 3
AUC=0.634504, best_iter=159
[top500_ranked] fold 4
AUC=0.634881, best_iter=110
[raw_clean] fold 0
AUC=0.624954, best_iter=127
[raw_clean] fold 1
AUC=0.639461, best_iter=208
[raw_clean] fold 2
AUC=0.626546, best_iter=152
[raw_clean] fold 3
AUC=0.642195, best_iter=169
[raw_clean] fold 4
AUC=0.639838, best_iter=233
[raw_clean_plus_meta] fold 0
AUC=0.624493, best_iter=101
[raw_clean_plus_meta] fold 1
AUC=0.649681, best_iter=273
[raw_clean_plus_meta] fold 2
AUC=0.615991, best_iter=149
[raw_clean_plus_meta] fold 3
AUC=0.643533, best_iter=125
[raw_clean_plus_meta] fold 4

,feature_set,n_cols,mean_auc,std_auc,mean_best_iteration
6,compact_for_linear_or_stack,965,0.645022,0.008539,244.6
4,top500_plus_engineered,1326,0.641231,0.003888,144.8
0,top300_ranked,300,0.638175,0.005746,162.2
3,raw_clean_plus_meta,1892,0.635860,0.013181,161.2
5,all_engineered,2186,0.635257,0.009286,203.2
2,raw_clean,1360,0.634599,0.007303,177.8
1,top500_ranked,500,0.633337,0.005809,146.2


## 23. Как использовать результаты дальше

После выполнения ноутбука будут готовы файлы в `processed_data/`:

- `X_train_raw_clean.*`
- `X_test_raw_clean.*`
- `X_train_engineered.*`
- `X_test_engineered.*`
- `feature_sets.json`
- `feature_ranking_combined.csv`
- `feature_profile_full.csv`
- `discovered_peak_values.csv`
- `test_train_exact_overlap.csv`

Дальше в modeling-ноутбуках можно делать так:

```python
import json
import pandas as pd
from pathlib import Path

processed = Path("processed_data")
feature_sets = json.load(open(processed / "feature_sets.json"))

X_raw = pd.read_parquet(processed / "X_train_raw_clean.parquet")
X_eng = pd.read_parquet(processed / "X_train_engineered.parquet")
y = pd.read_csv(processed / "y_train.csv")["target"]

cols = feature_sets["top500_plus_engineered"]
raw_cols = [c for c in cols if c in X_raw.columns]
eng_cols = [c for c in cols if c in X_eng.columns]
X = pd.concat([X_raw[raw_cols], X_eng[eng_cols]], axis=1)
```

Для ансамбля я бы проверял минимум:

1. `top500_ranked` — компактный сильный baseline;
2. `raw_clean` — все исходные хорошие признаки;
3. `raw_clean_plus_meta` — проверка пользы row/block stats;
4. `top500_plus_engineered` — главный кандидат;
5. `all_engineered` — максимальный набор под AutoML/LightGBM;
6. `compact_for_linear_or_stack` — для логистической регрессии / MLP / stacker.

## 24. Что можно добавить, если останется время

Широкий спектр следующих экспериментов:

1. **Noise-aware training**: получить OOF-предсказания, найти строки, где модель уверенно спорит с target, и не удалять их, а давать меньший `sample_weight`.
2. **Adversarial validation**: обучить модель отличать train от test. Если AUC высокий, есть distribution shift; drifty-фичи можно давать отдельным моделям или убирать из части ансамбля.
3. **RankGauss / QuantileTransformer** для линейных моделей и нейросетей.
4. **Autoencoder / MLP embeddings** по compact feature set.
5. **More interactions**: но только после того, как видно, какие топовые признаки реально стабильны.
6. **Separate models by segment**: например, отдельные модели для строк с высоким/низким `all_nonzero_count`.
7. **Pseudo-labeling**: осторожно добавить test-строки, где ансамбль очень уверен. Для дисбаланса это рискованно, только после стабильной CV.
8. **Feature selection through Optuna**: не `True/False` для каждой колонки, а выбор `top_k`, блоков и типов engineered-признаков.